# Pulsar Lighthouse — Scientific Visualization MVP

## Goal

Create a short seamless scientific-style animation of a rotating pulsar / neutron star with:

* rotating magnetic beams
* glow and volumetric feeling
* subtle particle motion
* HUD-friendly composition
* transparent or dark background compatibility
* export to GIF/WebM

The animation is intended as:

* standalone scientific visualization
* reusable HUD layer
* overlay element for future Nebulacast / Stellar Attractor interfaces

---

# Visual Concept

## Scene

A rapidly rotating neutron star is suspended in space.

The magnetic axis is tilted relative to the rotation axis.
Two radiation beams sweep through space like lighthouse beams.

The camera slowly drifts around the pulsar.

The star surface contains:

* animated emissive texture
* subtle plasma motion
* polar glow
* energetic atmosphere

The beams should:

* pulse slightly
* vary in intensity
* contain layered transparency
* illuminate surrounding dust particles

---

# Artistic Direction

## Style

Scientific cinematic visualization.

NOT:

* arcade game
* neon cyberpunk
* cartoon
* abstract VJ loop

YES:

* observatory visualization
* ESA / NASA style
* physically inspired
* restrained colors
* cinematic motion

---

# Technical Constraints

## Duration

6–12 seconds.

## Loop

Preferred seamless loop.

## Aspect Ratio

16:9

## Resolution

Initial MVP:

1280x720

Future:

1920x1080
4K

## Framerate

20–30 FPS

## Export Formats

* GIF
* WebM
* PNG frame sequence

---

# Architecture

## Rendering Strategy

Avoid real astrophysical simulation.

Use:

* procedural animation
* parametric geometry
* sprite-based particles
* layered alpha blending
* mathematical rotation transforms

This gives:

* stability
* speed
* artistic control
* deterministic rendering

---

# Scene Components

## 1. Neutron Star Core

### Requirements

* sphere rendering
* emissive shading
* rotating surface texture
* polar hotspots
* animated glow halo

### Simplification

Use 2D projected sphere illusion instead of full raytracing.

---

## 2. Rotation Axis

Optional thin semi-transparent axis line.

Useful for educational mode.

Must be switchable.

---

## 3. Magnetic Axis

Tilted relative to spin axis.

Controls:

* tilt angle
* beam width
* beam length
* beam intensity

---

## 4. Radiation Beams

### Behavior

* rotate with pulsar
* additive blending
* soft cone appearance
* layered gradients

### Rendering Strategy

Use:

* alpha gradients
* multiple translucent cones
* gaussian blur layers

NOT volumetric raymarching.

---

## 5. Particle Field

Small drifting particles:

* subtle
* sparse
* depth illusion
* low opacity

Purpose:

* scale perception
* cinematic depth
* beam interaction

---

## 6. Camera Motion

Very slow orbital drift.

Avoid:

* fast rotation
* shaking
* aggressive movement

Goal:

scientific documentary feeling.

---

# Animation Logic

## Pulsar Rotation

Constant angular velocity.

Example:

omega = 2π / period

MVP may use visually pleasing period instead of realistic millisecond pulsar timing.

---

## Beam Pulse

Low amplitude sinusoidal intensity modulation.

Example:

I(t) = I0 * (0.9 + 0.1 * sin(ωt))

---

## Surface Motion

Texture scrolling or procedural turbulence.

Should remain subtle.

---

# Recommended Stack

## MVP

Python
matplotlib
numpy
Pillow
imageio

---

## Advanced Version

Possible future migration:

* moderngl
* vispy
* pygfx
* blender python
* three.js/webgl

---

# Notebook Structure

## Cell 1

Imports + config.

---

## Cell 2

Scene parameters.

Examples:

* pulsar radius
* beam angle
* rotation speed
* particle count
* camera drift
* frame count

---

## Cell 3

Utility functions.

Examples:

* rotation matrices
* glow renderer
* particle projection
* alpha gradients

---

## Cell 4

Frame renderer.

Single frame generation.

---

## Cell 5

Animation loop.

Render all frames.

---

## Cell 6

GIF/WebM export.

Save into:

media-site/animations/

---

# MVP Acceptance Criteria

## Visual

* recognizable pulsar
* rotating beams
* cinematic appearance
* smooth motion
* no obvious artifacts

## Technical

* fully self-contained notebook
* no external assets required
* deterministic rendering
* export works locally

## Reusability

* modular parameters
* easy color replacement
* easy HUD integration
* transparent-background future support

---

# Future Extensions

## Phase 2

* relativistic beaming
* gravitational lensing
* synchrotron arcs
* magnetic field lines
* plasma torus
* shock structures

## Phase 3

* real-time WebGL version
* interactive HUD widget
* live parameter controls
* overlay mode
* integration into Observer Console

---

# Important Design Principle

This project is NOT an astrophysical simulator.

It is:

scientifically inspired cinematic visualization.

The priority order is:

1. visual clarity
2. stability
3. cinematic readability
4. physical plausibility
5. numerical realism


In [21]:
import os
import numpy as np
import imageio.v2 as imageio
from PIL import Image, ImageFilter

# ============================================================
# Pulsar Lighthouse — MVP animation
# One-cell self-contained notebook version
# ============================================================

# ----------------------------
# Config
# ----------------------------

OUT_DIR = os.path.join("media-site/animations", "pulsar_lighthouse")
OUT_FILE = os.path.join(OUT_DIR, "pulsar_lighthouse.gif")
os.makedirs(OUT_DIR, exist_ok=True)

WIDTH = 960
HEIGHT = 540
FPS = 24
DURATION = 8
N_FRAMES = FPS * DURATION

BACKGROUND = (0, 0, 0, 255)

STAR_RADIUS = 54
BEAM_LENGTH = 390
BEAM_WIDTH = 34
BEAM_TILT = np.deg2rad(28)

PARTICLE_COUNT = 420
RNG_SEED = 42

SHOW_AXIS = True

os.makedirs(OUT_DIR, exist_ok=True)
rng = np.random.default_rng(RNG_SEED)


# ----------------------------
# Helpers
# ----------------------------

def rgba_layer():
    return Image.new("RGBA", (WIDTH, HEIGHT), (0, 0, 0, 0))


def add_blurred_circle(layer, cx, cy, radius, color, blur=12):
    yy, xx = np.mgrid[0:HEIGHT, 0:WIDTH]
    d = np.sqrt((xx - cx) ** 2 + (yy - cy) ** 2)
    alpha = np.clip(1.0 - d / radius, 0, 1) ** 2

    arr = np.zeros((HEIGHT, WIDTH, 4), dtype=np.uint8)
    arr[..., 0] = color[0]
    arr[..., 1] = color[1]
    arr[..., 2] = color[2]
    arr[..., 3] = (alpha * color[3]).astype(np.uint8)

    img = Image.fromarray(arr, "RGBA").filter(ImageFilter.GaussianBlur(blur))
    layer.alpha_composite(img)


def draw_beam(layer, cx, cy, angle, length, width, color):
    yy, xx = np.mgrid[0:HEIGHT, 0:WIDTH]

    dx = xx - cx
    dy = yy - cy

    forward = dx * np.cos(angle) + dy * np.sin(angle)
    side = -dx * np.sin(angle) + dy * np.cos(angle)

    cone_width = width + forward * 0.13
    mask = (forward > 0) & (forward < length)

    radial = np.exp(-(side ** 2) / (2 * cone_width ** 2))
    fade = np.clip(1 - forward / length, 0, 1) ** 1.7
    alpha = radial * fade * mask

    arr = np.zeros((HEIGHT, WIDTH, 4), dtype=np.uint8)
    arr[..., 0] = color[0]
    arr[..., 1] = color[1]
    arr[..., 2] = color[2]
    arr[..., 3] = np.clip(alpha * color[3], 0, 255).astype(np.uint8)

    beam_img = Image.fromarray(arr, "RGBA").filter(ImageFilter.GaussianBlur(5))
    layer.alpha_composite(beam_img)


def draw_star(layer, cx, cy, phase):
    yy, xx = np.mgrid[0:HEIGHT, 0:WIDTH]
    dx = xx - cx
    dy = yy - cy
    r = np.sqrt(dx * dx + dy * dy)

    sphere = r <= STAR_RADIUS
    shade = np.clip(1 - r / STAR_RADIUS, 0, 1)

    texture = (
        0.5
        + 0.25 * np.sin(0.18 * dx + phase * 5)
        + 0.20 * np.sin(0.21 * dy - phase * 4)
        + 0.15 * np.sin(0.12 * (dx + dy) + phase * 7)
    )
    texture = np.clip(texture, 0, 1)

    glow = shade ** 0.55
    intensity = np.clip(0.35 + 0.65 * texture * glow, 0, 1)

    arr = np.zeros((HEIGHT, WIDTH, 4), dtype=np.uint8)

    arr[..., 0] = np.where(sphere, 80 + 120 * intensity, 0)
    arr[..., 1] = np.where(sphere, 150 + 90 * intensity, 0)
    arr[..., 2] = np.where(sphere, 220 + 35 * intensity, 0)
    arr[..., 3] = np.where(sphere, 255, 0)

    star_img = Image.fromarray(arr.astype(np.uint8), "RGBA")
    layer.alpha_composite(star_img)

    add_blurred_circle(layer, cx, cy, STAR_RADIUS * 2.4, (70, 170, 255, 95), blur=18)
    add_blurred_circle(layer, cx, cy, STAR_RADIUS * 1.3, (180, 230, 255, 110), blur=8)


def draw_particles(layer, particles, phase, beam_angle):
    arr = np.array(layer)

    for x0, y0, z, speed, size in particles:
        x = (x0 + phase * speed * 80) % WIDTH
        y = y0 + 8 * np.sin(phase * 2 + x0 * 0.01)

        cx = WIDTH * 0.5
        cy = HEIGHT * 0.5

        dx = x - cx
        dy = y - cy

        alignment = dx * np.cos(beam_angle) + dy * np.sin(beam_angle)
        side = abs(-dx * np.sin(beam_angle) + dy * np.cos(beam_angle))

        lit = alignment > 0 and alignment < BEAM_LENGTH and side < BEAM_WIDTH + alignment * 0.13
        alpha = 70 if lit else 28

        ix = int(x)
        iy = int(y)

        if 2 <= ix < WIDTH - 2 and 2 <= iy < HEIGHT - 2:
            s = int(size)
            arr[iy-s:iy+s+1, ix-s:ix+s+1, :] = [120, 170, 255, alpha]

    new_layer = Image.fromarray(arr.astype(np.uint8), "RGBA")
    layer.paste(new_layer)


from PIL import ImageDraw

def draw_axis(layer, cx, cy, phase):
    if not SHOW_AXIS:
        return

    axis_angle = -np.pi / 2 + 0.08 * np.sin(phase * 2 * np.pi)
    length = 170

    x1 = cx - length * np.cos(axis_angle)
    y1 = cy - length * np.sin(axis_angle)
    x2 = cx + length * np.cos(axis_angle)
    y2 = cy + length * np.sin(axis_angle)

    d = ImageDraw.Draw(layer)
    d.line(
        [(x1, y1), (x2, y2)],
        fill=(140, 210, 255, 110),
        width=1,
    )


# ----------------------------
# Particle initialization
# ----------------------------

particles = []
for _ in range(PARTICLE_COUNT):
    particles.append((
        rng.uniform(0, WIDTH),
        rng.uniform(0, HEIGHT),
        rng.uniform(0.2, 1.0),
        rng.uniform(0.15, 0.65),
        rng.integers(1, 2)
    ))


# ----------------------------
# Frame renderer
# ----------------------------

def render_frame(i):
    phase = i / N_FRAMES
    t = phase * 2 * np.pi

    bg = Image.new("RGBA", (WIDTH, HEIGHT), BACKGROUND)

    layer = rgba_layer()

    cx = WIDTH * 0.5 + 16 * np.sin(t * 0.35)
    cy = HEIGHT * 0.5 + 8 * np.cos(t * 0.28)

    spin = t * 3.0
    projected_angle = spin
    beam_visibility = 0.55 + 0.45 * np.cos(spin)

    beam_alpha = int(115 + 75 * abs(beam_visibility))

    beam_angle_1 = projected_angle + BEAM_TILT
    beam_angle_2 = beam_angle_1 + np.pi

    draw_particles(layer, particles, phase, beam_angle_1)

    draw_beam(layer, cx, cy, beam_angle_1, BEAM_LENGTH, BEAM_WIDTH, (90, 190, 255, beam_alpha))
    draw_beam(layer, cx, cy, beam_angle_2, BEAM_LENGTH, BEAM_WIDTH, (90, 190, 255, int(beam_alpha * 0.65)))

    draw_axis(layer, cx, cy, phase)

    draw_star(layer, cx, cy, t)

    bg.alpha_composite(layer)

    return np.array(bg.convert("RGB"))


# ----------------------------
# Render animation
# ----------------------------

frames = []

for i in range(N_FRAMES):
    frames.append(render_frame(i))

imageio.mimsave(
    OUT_FILE,
    frames,
    fps=FPS,
    loop=0
)

print(f"Saved: {OUT_FILE}")

/var/folders/_b/cfj7mly10r9f3nkywrlqnl300000gn/T/ipykernel_1765/2631665930.py:150: DeprecationWarning: 'mode' parameter is deprecated and will be removed in Pillow 13 (2026-10-15)
  new_layer = Image.fromarray(arr.astype(np.uint8), "RGBA")
/var/folders/_b/cfj7mly10r9f3nkywrlqnl300000gn/T/ipykernel_1765/2631665930.py:86: DeprecationWarning: 'mode' parameter is deprecated and will be removed in Pillow 13 (2026-10-15)
  beam_img = Image.fromarray(arr, "RGBA").filter(ImageFilter.GaussianBlur(5))
/var/folders/_b/cfj7mly10r9f3nkywrlqnl300000gn/T/ipykernel_1765/2631665930.py:117: DeprecationWarning: 'mode' parameter is deprecated and will be removed in Pillow 13 (2026-10-15)
  star_img = Image.fromarray(arr.astype(np.uint8), "RGBA")
/var/folders/_b/cfj7mly10r9f3nkywrlqnl300000gn/T/ipykernel_1765/2631665930.py:60: DeprecationWarning: 'mode' parameter is deprecated and will be removed in Pillow 13 (2026-10-15)
  img = Image.fromarray(arr, "RGBA").filter(ImageFilter.GaussianBlur(blur))


Saved: animations/pulsar_lighthouse/pulsar_lighthouse.gif


# Gravitational Wave Grid

In [22]:
from __future__ import annotations















from pathlib import Path







import numpy as np







import imageio.v2 as imageio







from PIL import Image, ImageDraw, ImageFilter























# ============================================================







# Gravitational Wave Grid — MVP animation







# One-cell self-contained notebook version







# ============================================================















OUTPUT_FORMAT = "webm"  # "gif" | "mp4"







FPS = 24







DURATION = 8







TOTAL_FRAMES = FPS * DURATION















WIDTH = 960







HEIGHT = 540















ANIMATION_NAME = "gravitational_wave_grid"







OUT_DIR = Path("media-site/animations") / ANIMATION_NAME







OUT_DIR.mkdir(parents=True, exist_ok=True)















OUT_FILE = OUT_DIR / f"{ANIMATION_NAME}.{OUTPUT_FORMAT}"















BG = (0, 0, 0, 255)















GRID_N = 46







GRID_SIZE = 9.0















CAMERA_DISTANCE = 11.0







CAMERA_ELEVATION = np.deg2rad(34)







CAMERA_ORBIT_SPEED = 0.35















MASS_ORBIT_RADIUS_INITIAL = 2.2







MASS_ORBIT_RADIUS_FINAL = 0.34















MERGER_PHASE = 0.62















WAVE_SPEED = 9.5







WAVE_AMPLITUDE = 0.85







WAVE_DECAY = 0.24















STAR_COUNT = 450







RNG_SEED = 42























# ============================================================







# HELPERS







# ============================================================















def smoothstep(t: float) -> float:







    t = float(np.clip(t, 0.0, 1.0))







    return t * t * (3.0 - 2.0 * t)























def rgba_layer() -> Image.Image:







    return Image.new("RGBA", (WIDTH, HEIGHT), (0, 0, 0, 0))























def add_glow(base: Image.Image, layer: Image.Image, blur: int = 6):







    glow = layer.filter(ImageFilter.GaussianBlur(blur))







    base.alpha_composite(glow)







    base.alpha_composite(layer)























def rotate_points(points: np.ndarray, yaw: float, pitch: float) -> np.ndarray:







    cy, sy = np.cos(yaw), np.sin(yaw)







    cp, sp = np.cos(pitch), np.sin(pitch)















    ry = np.array([







        [cy, -sy, 0.0],







        [sy,  cy, 0.0],







        [0.0, 0.0, 1.0],







    ])















    rx = np.array([







        [1.0, 0.0, 0.0],







        [0.0, cp, -sp],







        [0.0, sp,  cp],







    ])















    return points @ ry.T @ rx.T























def project(points: np.ndarray, yaw: float, pitch: float) -> np.ndarray:







    p = rotate_points(points, yaw, pitch)















    z = p[:, 2] + CAMERA_DISTANCE







    scale = 440 / z















    x2 = WIDTH * 0.5 + p[:, 0] * scale







    y2 = HEIGHT * 0.56 - p[:, 1] * scale















    return np.column_stack([x2, y2, z])























def grid_height(x: np.ndarray, y: np.ndarray, phase: float) -> np.ndarray:







    r = np.sqrt(x * x + y * y)















    inspiral = smoothstep(phase / MERGER_PHASE)







    merger = smoothstep((phase - MERGER_PHASE) / 0.18)







    ringdown = smoothstep((phase - 0.72) / 0.25)















    masses = binary_positions_3d(phase)















    if len(masses) == 2:







        m1 = masses[0][:2]







        m2 = masses[1][:2]















        d1 = np.sqrt((x - m1[0]) ** 2 + (y - m1[1]) ** 2)







        d2 = np.sqrt((x - m2[0]) ** 2 + (y - m2[1]) ** 2)















        # Positive z means visually downward in current projection.







        well1 = 0.85 / (d1 * d1 + 0.28)







        well2 = 0.85 / (d2 * d2 + 0.28)















        wells = well1 + well2







    else:







        m = masses[0][:2]







        d = np.sqrt((x - m[0]) ** 2 + (y - m[1]) ** 2)















        wells = 1.35 / (d * d + 0.38)























    # Outgoing gravitational wave after merger.







    wave_age = max(0.0, phase - MERGER_PHASE)







    wave_front = wave_age * WAVE_SPEED















    wave = (







        WAVE_AMPLITUDE







        * np.sin(8.0 * (r - wave_front))







        * np.exp(-WAVE_DECAY * r)







        * np.exp(-1.6 * np.abs(r - wave_front))







        * merger







        * (1.0 - 0.35 * ringdown)







    )















    # Ringdown close to center.







    ring = (







        0.35







        * np.sin(18.0 * phase + 5.0 * r)







        * np.exp(-0.7 * r)







        * ringdown







    )















    return wells * (1.0 - merger) + wave + ring















def make_grid_lines(phase: float):







    coords = np.linspace(-GRID_SIZE, GRID_SIZE, GRID_N)















    lines = []















    for x in coords:







        y = np.linspace(-GRID_SIZE, GRID_SIZE, 220)







        xx = np.full_like(y, x)







        zz = grid_height(xx, y, phase)







        lines.append(np.column_stack([xx, y, zz]))















    for y in coords:







        x = np.linspace(-GRID_SIZE, GRID_SIZE, 220)







        yy = np.full_like(x, y)







        zz = grid_height(x, yy, phase)







        lines.append(np.column_stack([x, yy, zz]))















    return lines























def draw_starfield(base: Image.Image, phase: float):







    rng = np.random.default_rng(RNG_SEED)







    d = ImageDraw.Draw(base)















    for _ in range(STAR_COUNT):







        x = rng.uniform(0, WIDTH)







        y = rng.uniform(0, HEIGHT)







        size = rng.uniform(0.6, 1.8)







        twinkle = 0.55 + 0.45 * np.sin(phase * 2 * np.pi * rng.uniform(0.5, 2.0) + rng.uniform(0, 6.28))







        alpha = int(rng.uniform(45, 130) * twinkle)















        d.ellipse(







            [x - size, y - size, x + size, y + size],







            fill=(190, 220, 255, alpha),







        )























def draw_grid(base: Image.Image, phase: float):







    layer = rgba_layer()







    d = ImageDraw.Draw(layer)















    yaw = 2 * np.pi * CAMERA_ORBIT_SPEED * phase







    pitch = CAMERA_ELEVATION















    lines = make_grid_lines(phase)















    for line in lines:







        projected = project(line, yaw, pitch)















        pts = [(float(x), float(y)) for x, y, _z in projected]















        # Depth cue.







        mean_z = np.mean(projected[:, 2])







        depth = np.clip((CAMERA_DISTANCE + 7.0 - mean_z) / 10.0, 0.2, 1.0)







        alpha = int(55 + 120 * depth)















        d.line(







            pts,







            fill=(90, 220, 255, alpha),







            width=1,







            joint="curve",







        )















    add_glow(base, layer, blur=4)























def binary_positions_3d(phase: float):







    inspiral = smoothstep(phase / MERGER_PHASE)















    sep = MASS_ORBIT_RADIUS_FINAL + (







        MASS_ORBIT_RADIUS_INITIAL - MASS_ORBIT_RADIUS_FINAL







    ) * (1.0 - inspiral) ** 1.85















    spinup = 1.0 + 3.8 * inspiral ** 2.2







    angle = 2 * np.pi * (phase * spinup * 2.2)















    if phase < MERGER_PHASE:







        return [







            np.array([ sep * np.cos(angle),  sep * np.sin(angle), 0.45]),







            np.array([-sep * np.cos(angle), -sep * np.sin(angle), 0.45]),







        ]















    ringdown = smoothstep((phase - MERGER_PHASE) / 0.26)















    wobble_amp = 0.18 * (1.0 - ringdown)







    wobble = wobble_amp * np.array([







        np.cos(angle * 2.3),







        np.sin(angle * 2.3),







        0.15 * np.sin(angle * 3.1),







    ])















    return [np.array([0.0, 0.0, 0.55]) + wobble]























def mass_positions(phase: float):







    return binary_positions_3d(phase)























def draw_masses(base: Image.Image, phase: float):







    layer = rgba_layer()







    d = ImageDraw.Draw(layer)















    yaw = 2 * np.pi * CAMERA_ORBIT_SPEED * phase







    pitch = CAMERA_ELEVATION















    merger_window = 0.045







    merger_t = smoothstep((phase - (MERGER_PHASE - merger_window)) / (merger_window * 1.45))







    ringdown_t = smoothstep((phase - MERGER_PHASE) / 0.30)















    if phase < MERGER_PHASE + 0.035:







        inspiral = smoothstep(phase / MERGER_PHASE)















        sep = MASS_ORBIT_RADIUS_FINAL + (







            MASS_ORBIT_RADIUS_INITIAL - MASS_ORBIT_RADIUS_FINAL







        ) * (1.0 - inspiral) ** 1.85















        spinup = 1.0 + 3.8 * inspiral ** 2.2







        angle = 2 * np.pi * (phase * spinup * 2.2)















        p1 = np.array([sep * np.cos(angle), sep * np.sin(angle), 0.45])







        p2 = np.array([-sep * np.cos(angle), -sep * np.sin(angle), 0.45])















        projected = project(np.array([p1, p2]), yaw, pitch)







        p1s = projected[0]







        p2s = projected[1]















        cx = (p1s[0] + p2s[0]) * 0.5







        cy = (p1s[1] + p2s[1]) * 0.5















        if merger_t > 0.52:







            yy, xx = np.mgrid[0:HEIGHT, 0:WIDTH]







            field = np.zeros((HEIGHT, WIDTH), dtype=np.float32)















            for px, py in [(p1s[0], p1s[1]), (p2s[0], p2s[1])]:







                dx = xx - px







                dy = yy - py







                rr2 = dx * dx + dy * dy







                field += 90000.0 / (rr2 + 1.0)















            threshold = 42 + 24 * (1.0 - merger_t)







            mask = field > threshold















            shadow_arr = np.zeros((HEIGHT, WIDTH, 4), dtype=np.uint8)







            shadow_arr[..., 0] = 3







            shadow_arr[..., 1] = 5







            shadow_arr[..., 2] = 8







            shadow_arr[..., 3] = np.where(mask, 185, 0)















            shadow = Image.fromarray(shadow_arr, "RGBA")







            layer.alpha_composite(shadow)















            edge = (field > threshold * 0.92) & (field < threshold * 1.12)















            ring_arr = np.zeros((HEIGHT, WIDTH, 4), dtype=np.uint8)







            ring_arr[..., 0] = 235







            ring_arr[..., 1] = 255







            ring_arr[..., 2] = 255







            ring_arr[..., 3] = np.where(edge, 230, 0)















            ring_img = Image.fromarray(ring_arr, "RGBA").filter(







                ImageFilter.GaussianBlur(1.0)







            )







            layer.alpha_composite(ring_img)















            for rr, alpha in [







                (92, 24),







                (58, 46),







                (32, 90),







            ]:







                d.ellipse(







                    [cx - rr, cy - rr, cx + rr, cy + rr],







                    fill=(130, 220, 255, int(alpha * merger_t)),







                )















        else:







            for x, y, _z in projected:







                r = 18 + 12 * merger_t















                for scale, alpha in [







                    (3.7, 22),







                    (2.2, 50),







                    (1.35, 95),







                ]:







                    rr = r * scale







                    d.ellipse(







                        [x - rr, y - rr, x + rr, y + rr],







                        fill=(120, 210, 255, int(alpha * (1.0 + 0.35 * merger_t))),







                    )















                d.ellipse(







                    [x - r, y - r, x + r, y + r],







                    fill=(3, 5, 8, 185),







                )















                d.ellipse(







                    [







                        x - r * 1.12,







                        y - r * 1.12,







                        x + r * 1.12,







                        y + r * 1.12,







                    ],







                    outline=(235, 255, 255, 235),







                    width=2,







                )















        add_glow(base, layer, blur=6)







        return















    p = mass_positions(phase)[0]







    x, y, _z = project(np.array([p]), yaw, pitch)[0]















    pulse = (1.0 - ringdown_t) * np.sin(







        2 * np.pi * (phase - MERGER_PHASE) * 18.0







    ) ** 2















    r = 26 + 10 * pulse















    for scale, alpha in [







        (4.2, 20),







        (2.6, 44),







        (1.55, 82),







    ]:







        rr = r * scale







        d.ellipse(







            [x - rr, y - rr, x + rr, y + rr],







            fill=(130, 220, 255, int(alpha * (1.0 - 0.25 * ringdown_t))),







        )















    d.ellipse(







        [x - r * 1.45, y - r * 0.92, x + r * 1.45, y + r * 0.92],







        outline=(210, 250, 255, int(210 * (0.7 + 0.3 * ringdown_t))),







        width=2,







    )















    d.ellipse(







        [x - r, y - r, x + r, y + r],







        fill=(3, 5, 8, 190),







    )















    d.ellipse(







        [x - r * 1.12, y - r * 1.12, x + r * 1.12, y + r * 1.12],







        outline=(235, 255, 255, 230),







        width=2,







    )















    add_glow(base, layer, blur=6)























    # --------------------------------------------------------







    # AFTER MERGER: single remnant + ringdown pulsation







    # --------------------------------------------------------















    p = mass_positions(phase)[0]







    x, y, _z = project(np.array([p]), yaw, pitch)[0]















    pulse = (1.0 - ringdown_t) * np.sin(2 * np.pi * (phase - MERGER_PHASE) * 18.0) ** 2







    r = 26 + 10 * pulse















    for scale, alpha in [







        (4.2, 20),







        (2.6, 44),







        (1.55, 82),







    ]:







        rr = r * scale







        d.ellipse(







            [x - rr, y - rr, x + rr, y + rr],







            fill=(130, 220, 255, int(alpha * (1.0 - 0.25 * ringdown_t))),







        )















    # Flattened photon-ring-like outline.







    d.ellipse(







        [x - r * 1.45, y - r * 0.92, x + r * 1.45, y + r * 0.92],







        outline=(210, 250, 255, int(210 * (0.7 + 0.3 * ringdown_t))),







        width=2,







    )















    # Semi-transparent remnant shadow.







    d.ellipse(







        [x - r, y - r, x + r, y + r],







        fill=(3, 5, 8, 190),







    )















    # Final remnant photon ring.







    d.ellipse(







        [x - r * 1.12, y - r * 1.12, x + r * 1.12, y + r * 1.12],







        outline=(235, 255, 255, 230),







        width=2,







    )















    add_glow(base, layer, blur=6)























def draw_wavefront(base: Image.Image, phase: float):







    if phase < MERGER_PHASE:







        return















    wave_age = phase - MERGER_PHASE















    layer = rgba_layer()







    d = ImageDraw.Draw(layer)















    yaw = 2 * np.pi * CAMERA_ORBIT_SPEED * phase







    pitch = CAMERA_ELEVATION















    # Несколько расходящихся импульсов, а не одно кольцо.







    pulses = [







        (0.00, 1.00, 2),







        (0.08, 0.55, 1),







        (0.16, 0.32, 1),







    ]















    for delay, strength, width in pulses:







        local_age = wave_age - delay







        if local_age <= 0:







            continue















        radius = local_age * WAVE_SPEED















        if radius <= 0.05 or radius > GRID_SIZE * 1.35:







            continue















        t = np.linspace(0, 2 * np.pi, 620)















        # Кольцо чуть деформировано, чтобы не выглядело как идеальный HUD-circle.







        rr = radius * (







            1.0







            + 0.045 * np.sin(2 * t - phase * 14)







            + 0.025 * np.sin(5 * t + phase * 9)







        )















        x = rr * np.cos(t)







        y = rr * np.sin(t)







        z = 0.72 + 0.16 * np.sin(8 * t - phase * 20)















        pts3 = np.column_stack([x, y, z])







        pts2 = project(pts3, yaw, pitch)















        pts = [(float(px), float(py)) for px, py, _ in pts2]















        alpha = int(235 * strength * np.exp(-0.12 * radius))







        alpha = max(0, min(235, alpha))















        if alpha > 4:







            d.line(







                pts,







                fill=(185, 255, 255, alpha),







                width=width,







                joint="curve",







            )















    add_glow(base, layer, blur=9)























def render_frame(i: int) -> np.ndarray:







    phase = i / TOTAL_FRAMES















    base = Image.new("RGBA", (WIDTH, HEIGHT), BG)















    draw_starfield(base, phase)







    draw_grid(base, phase)







    draw_wavefront(base, phase)







    draw_masses(base, phase)















    return np.array(base.convert("RGB"))























# ============================================================







# RENDER







# ============================================================















frames = []















print("[START] gravitational wave grid")







print(f"[CONFIG] frames={TOTAL_FRAMES}, fps={FPS}, output={OUT_FILE}")















for i in range(TOTAL_FRAMES):







    if i % FPS == 0:







        print(f"[RENDER] frame {i}/{TOTAL_FRAMES}")







    frames.append(render_frame(i))















if OUTPUT_FORMAT == "gif":







    imageio.mimsave(







        OUT_FILE,







        frames,







        fps=FPS,







        loop=0,







    )







elif OUTPUT_FORMAT == "webm":







    from vizlib.animation_export import export_animation







    export_animation(frames, OUT_DIR, ANIMATION_NAME, "webm", FPS)







    OUT_FILE = OUT_DIR / f"{ANIMATION_NAME}.webm"







else:







    imageio.mimsave(







        OUT_FILE,







        frames,







        fps=FPS,







        quality=9,







        macro_block_size=1,







    )















print(f"[SAVED] {OUT_FILE.resolve()}")





[START] gravitational wave grid
[CONFIG] frames=192, fps=24, output=animations/gravitational_wave_grid/gravitational_wave_grid.gif
[RENDER] frame 0/192
[RENDER] frame 24/192
[RENDER] frame 48/192
[RENDER] frame 72/192
[RENDER] frame 96/192


/var/folders/_b/cfj7mly10r9f3nkywrlqnl300000gn/T/ipykernel_1765/3497821699.py:299: DeprecationWarning: 'mode' parameter is deprecated and will be removed in Pillow 13 (2026-10-15)
  shadow = Image.fromarray(shadow_arr, "RGBA")
/var/folders/_b/cfj7mly10r9f3nkywrlqnl300000gn/T/ipykernel_1765/3497821699.py:310: DeprecationWarning: 'mode' parameter is deprecated and will be removed in Pillow 13 (2026-10-15)
  ring_img = Image.fromarray(ring_arr, "RGBA").filter(


[RENDER] frame 120/192
[RENDER] frame 144/192
[RENDER] frame 168/192
[SAVED] /Users/mloktionov/PycharmProjects/Stellar_Attractor/ANIM/Infographics/Telemetry/animations/gravitational_wave_grid/gravitational_wave_grid.gif


# Wormhole / Einstein–Rosen bridge.

In [23]:
from __future__ import annotations

from pathlib import Path
import numpy as np
import imageio.v2 as imageio
from PIL import Image, ImageDraw, ImageFilter


# ============================================================
# Wormhole / Einstein-Rosen Bridge — MVP animation
# One-cell self-contained notebook version
# ============================================================

OUTPUT_FORMAT = "gif"  # "gif" | "mp4"
FPS = 24
DURATION = 8
TOTAL_FRAMES = FPS * DURATION

WIDTH = 960
HEIGHT = 540

ANIMATION_NAME = "wormhole_bridge"
OUT_DIR = Path("media-site/animations") / ANIMATION_NAME
OUT_DIR.mkdir(parents=True, exist_ok=True)
OUT_FILE = OUT_DIR / f"{ANIMATION_NAME}.{OUTPUT_FORMAT}"

BG = (0, 0, 0, 255)

GRID_SIZE = 8.5
GRID_N = 42
GRID_SAMPLES = 220

CAMERA_DISTANCE = 12.0
CAMERA_ELEVATION = np.deg2rad(34)
CAMERA_ORBIT_SPEED = 0.28

THROAT_RADIUS = 1.15
WELL_DEPTH = 4.2
LIGHT_RAY_COUNT = 34

STAR_COUNT = 500
RNG_SEED = 42


# ============================================================
# HELPERS
# ============================================================

def rgba_layer() -> Image.Image:
    return Image.new("RGBA", (WIDTH, HEIGHT), (0, 0, 0, 0))


def add_glow(base: Image.Image, layer: Image.Image, blur: int = 5):
    glow = layer.filter(ImageFilter.GaussianBlur(blur))
    base.alpha_composite(glow)
    base.alpha_composite(layer)


def smoothstep(t: float) -> float:
    t = float(np.clip(t, 0.0, 1.0))
    return t * t * (3 - 2 * t)


def rotate_points(points: np.ndarray, yaw: float, pitch: float) -> np.ndarray:
    cy, sy = np.cos(yaw), np.sin(yaw)
    cp, sp = np.cos(pitch), np.sin(pitch)

    rz = np.array([
        [cy, -sy, 0.0],
        [sy,  cy, 0.0],
        [0.0, 0.0, 1.0],
    ])

    rx = np.array([
        [1.0, 0.0, 0.0],
        [0.0, cp, -sp],
        [0.0, sp,  cp],
    ])

    return points @ rz.T @ rx.T


def project(points: np.ndarray, yaw: float, pitch: float) -> np.ndarray:
    p = rotate_points(points, yaw, pitch)

    z = p[:, 2] + CAMERA_DISTANCE
    scale = 430 / z

    x2 = WIDTH * 0.5 + p[:, 0] * scale
    y2 = HEIGHT * 0.56 - p[:, 1] * scale

    return np.column_stack([x2, y2, z])


def wormhole_height(x: np.ndarray, y: np.ndarray, side: float, phase: float) -> np.ndarray:
    r = np.sqrt(x * x + y * y)

    throat = WELL_DEPTH / (r * r + THROAT_RADIUS ** 2)
    ripple = (
        0.18
        * np.sin(5.5 * r - phase * 2 * np.pi * 2.0)
        * np.exp(-0.28 * r)
    )

    # Two sheets: upper and lower bridge.
    return side * (throat + ripple)


def make_sheet_lines(side: float, phase: float):
    coords = np.linspace(-GRID_SIZE, GRID_SIZE, GRID_N)
    lines = []

    for x in coords:
        y = np.linspace(-GRID_SIZE, GRID_SIZE, GRID_SAMPLES)
        xx = np.full_like(y, x)
        zz = wormhole_height(xx, y, side, phase)
        lines.append(np.column_stack([xx, y, zz]))

    for y in coords:
        x = np.linspace(-GRID_SIZE, GRID_SIZE, GRID_SAMPLES)
        yy = np.full_like(x, y)
        zz = wormhole_height(x, yy, side, phase)
        lines.append(np.column_stack([x, yy, zz]))

    return lines


def draw_starfield(base: Image.Image, phase: float):
    rng = np.random.default_rng(RNG_SEED)
    d = ImageDraw.Draw(base)

    for _ in range(STAR_COUNT):
        x = rng.uniform(0, WIDTH)
        y = rng.uniform(0, HEIGHT)
        size = rng.uniform(0.5, 1.7)
        twinkle = 0.55 + 0.45 * np.sin(
            phase * 2 * np.pi * rng.uniform(0.4, 1.8) + rng.uniform(0, 6.28)
        )
        alpha = int(rng.uniform(35, 120) * twinkle)

        d.ellipse(
            [x - size, y - size, x + size, y + size],
            fill=(185, 220, 255, alpha),
        )


def draw_grid_sheet(base: Image.Image, side: float, phase: float, color, alpha_scale: float):
    layer = rgba_layer()
    d = ImageDraw.Draw(layer)

    yaw = 2 * np.pi * CAMERA_ORBIT_SPEED * phase
    pitch = CAMERA_ELEVATION

    for line in make_sheet_lines(side, phase):
        pts3 = line
        pts2 = project(pts3, yaw, pitch)

        pts = [(float(x), float(y)) for x, y, _z in pts2]

        mean_z = np.mean(pts2[:, 2])
        depth = np.clip((CAMERA_DISTANCE + 7.0 - mean_z) / 12.0, 0.15, 1.0)
        alpha = int((45 + 130 * depth) * alpha_scale)

        d.line(
            pts,
            fill=(color[0], color[1], color[2], alpha),
            width=1,
            joint="curve",
        )

    add_glow(base, layer, blur=4)


def draw_throat(base: Image.Image, phase: float):
    layer = rgba_layer()
    d = ImageDraw.Draw(layer)

    yaw = 2 * np.pi * CAMERA_ORBIT_SPEED * phase
    pitch = CAMERA_ELEVATION

    t = np.linspace(0, 2 * np.pi, 720)
    r = THROAT_RADIUS * (1.0 + 0.035 * np.sin(6 * t + phase * 2 * np.pi * 2))

    x = r * np.cos(t)
    y = r * np.sin(t)
    z = 0.18 * np.sin(3 * t + phase * 2 * np.pi)

    pts3 = np.column_stack([x, y, z])
    pts2 = project(pts3, yaw, pitch)
    pts = [(float(px), float(py)) for px, py, _ in pts2]

    d.line(pts, fill=(220, 255, 255, 235), width=3, joint="curve")

    for scale, alpha in [(1.8, 36), (1.28, 70)]:
        rr = THROAT_RADIUS * scale
        xg = rr * np.cos(t)
        yg = rr * np.sin(t)
        zg = np.zeros_like(t)
        glow2 = project(np.column_stack([xg, yg, zg]), yaw, pitch)
        gpts = [(float(px), float(py)) for px, py, _ in glow2]
        d.line(gpts, fill=(90, 220, 255, alpha), width=2, joint="curve")

    add_glow(base, layer, blur=7)

def draw_solar_occlusion(base, phase, cx, cy):
    layer = rgba()

    yy, xx = np.mgrid[0:HEIGHT, 0:WIDTH]

    dx = xx - cx
    dy = yy - cy
    r = np.sqrt(dx * dx + dy * dy)

    sphere = r <= SUN_RADIUS

    shade = np.clip(1.0 - r / SUN_RADIUS, 0, 1)

    turbulence = (
        0.52
        + 0.22 * np.sin(dx * 0.045 + phase * 8)
        + 0.16 * np.sin(dy * 0.055 - phase * 10)
        + 0.12 * np.sin((dx + dy) * 0.03 + phase * 13)
    )

    turbulence = np.clip(turbulence, 0, 1)
    intensity = shade ** 0.42 * turbulence

    arr = np.zeros((HEIGHT, WIDTH, 4), dtype=np.uint8)

    arr[..., 0] = np.where(sphere, 255, 0)
    arr[..., 1] = np.where(sphere, 140 + 80 * intensity, 0)
    arr[..., 2] = np.where(sphere, 30 + 40 * intensity, 0)
    arr[..., 3] = np.where(sphere, 255, 0)

    img = Image.fromarray(arr.astype(np.uint8), "RGBA")
    layer.alpha_composite(img)

    base.alpha_composite(layer)

    
def draw_light_rays(base: Image.Image, phase: float):
    layer = rgba_layer()
    d = ImageDraw.Draw(layer)

    yaw = 2 * np.pi * CAMERA_ORBIT_SPEED * phase
    pitch = CAMERA_ELEVATION

    for k in range(LIGHT_RAY_COUNT):
        a = 2 * np.pi * k / LIGHT_RAY_COUNT
        offset = 0.35 * np.sin(phase * 2 * np.pi + k * 0.7)

        s = np.linspace(-1.0, 1.0, 260)

        r = THROAT_RADIUS * (1.12 + 0.12 * np.sin(k))
        theta = a + 0.9 * s + phase * 2 * np.pi * 0.9

        x = r * np.cos(theta) * (1.0 + 1.15 * np.abs(s))
        y = r * np.sin(theta) * (1.0 + 1.15 * np.abs(s))
        z = -WELL_DEPTH * s + offset * np.sin(np.pi * s)

        pts3 = np.column_stack([x, y, z])
        pts2 = project(pts3, yaw, pitch)

        pts = [(float(px), float(py)) for px, py, _ in pts2]

        alpha = 60 + int(70 * (0.5 + 0.5 * np.sin(phase * 2 * np.pi * 2 + k)))
        d.line(
            pts,
            fill=(180, 255, 255, alpha),
            width=1,
            joint="curve",
        )

    add_glow(base, layer, blur=5)


def draw_center_shadow(base: Image.Image, phase: float):
    layer = rgba_layer()
    d = ImageDraw.Draw(layer)

    yaw = 2 * np.pi * CAMERA_ORBIT_SPEED * phase
    pitch = CAMERA_ELEVATION

    center = project(np.array([[0.0, 0.0, 0.0]]), yaw, pitch)[0]
    x, y, _ = center

    pulse = 0.5 + 0.5 * np.sin(phase * 2 * np.pi * 2.0) ** 2

    for rr, alpha in [
        (92, 34),
        (58, 72),
        (34, 130),
    ]:
        rr = rr * (0.92 + 0.08 * pulse)
        d.ellipse(
            [x - rr, y - rr, x + rr, y + rr],
            fill=(5, 8, 16, alpha),
        )

    add_glow(base, layer, blur=8)


def render_frame(i):
    phase = i / TOTAL_FRAMES

    base = Image.new("RGBA", (WIDTH, HEIGHT), BG)

    draw_background(base, phase)

    cx, cy = draw_sun(base, phase)

    draw_loops(base, phase, cx, cy)
    draw_plasma(base, phase, cx, cy)
    draw_shockwave(base, phase, cx, cy)
    draw_cme(base, phase, cx, cy)

    # opaque photosphere mask on top:
    # hides far-side loops and particles behind the solar disk
    draw_solar_occlusion(base, phase, cx, cy)

    return np.array(base.convert("RGB"))


# ============================================================
# RENDER
# ============================================================

frames = []

print("[START] wormhole bridge")
print(f"[CONFIG] frames={TOTAL_FRAMES}, fps={FPS}, output={OUT_FILE}")

for i in range(TOTAL_FRAMES):
    if i % FPS == 0:
        print(f"[RENDER] frame {i}/{TOTAL_FRAMES}")
    frames.append(render_frame(i))

if OUTPUT_FORMAT == "gif":
    imageio.mimsave(
        OUT_FILE,
        frames,
        fps=FPS,
        loop=0,
    )
else:
    imageio.mimsave(
        OUT_FILE,
        frames,
        fps=FPS,
        quality=9,
        macro_block_size=1,
    )

print(f"[SAVED] {OUT_FILE.resolve()}")

[START] wormhole bridge
[CONFIG] frames=192, fps=24, output=animations/wormhole_bridge/wormhole_bridge.gif
[RENDER] frame 0/192


/var/folders/_b/cfj7mly10r9f3nkywrlqnl300000gn/T/ipykernel_1765/4083860716.py:138: DeprecationWarning: 'mode' parameter is deprecated and will be removed in Pillow 13 (2026-10-15)
  img = Image.fromarray(arr.astype(np.uint8), "RGBA")


ValueError: too many values to unpack (expected 3)

The animation visualizes a hypothetical Einstein–Rosen bridge — a wormhole connecting two distant regions of spacetime through a shared gravitational throat. Two curved spacetime sheets descend toward a luminous central passage, while streams of light spiral around the bridge and disappear into its interior. The slowly orbiting camera emphasizes the geometry of warped space, inspired by relativistic embeddings and classic science-fiction depictions of traversable wormholes.

# Relativistic jet / active galactic nucleus

In [ ]:
from __future__ import annotations

from pathlib import Path
import numpy as np
import imageio.v2 as imageio
from PIL import Image, ImageDraw, ImageFilter


# ============================================================
# Relativistic Jet / Active Galactic Nucleus
# Stylized cinematic visualization
# ============================================================

OUTPUT_FORMAT = "gif"  # gif | mp4
FPS = 24
DURATION = 8
TOTAL_FRAMES = FPS * DURATION

WIDTH = 960
HEIGHT = 540

ANIMATION_NAME = "relativistic_jet"
OUT_DIR = Path("media-site/animations") / ANIMATION_NAME
OUT_DIR.mkdir(parents=True, exist_ok=True)

OUT_FILE = OUT_DIR / f"{ANIMATION_NAME}.{OUTPUT_FORMAT}"

BG = (0, 0, 0, 255)

RNG_SEED = 42
STAR_COUNT = 420

DISK_R_INNER = 48
DISK_R_OUTER = 220

JET_LENGTH = 430
JET_WIDTH = 56

PARTICLE_COUNT = 1400

CAMERA_ORBIT_SPEED = 0.22
CAMERA_ELEVATION = np.deg2rad(28)

BH_RADIUS = 36


# ============================================================
# HELPERS
# ============================================================

def rgba_layer():
    return Image.new("RGBA", (WIDTH, HEIGHT), (0, 0, 0, 0))


def add_glow(base, layer, blur=6):
    glow = layer.filter(ImageFilter.GaussianBlur(blur))
    base.alpha_composite(glow)
    base.alpha_composite(layer)


def smoothstep(t):
    t = np.clip(t, 0.0, 1.0)
    return t * t * (3 - 2 * t)


def rotate(points, yaw, pitch):
    cy, sy = np.cos(yaw), np.sin(yaw)
    cp, sp = np.cos(pitch), np.sin(pitch)

    rz = np.array([
        [cy, -sy, 0],
        [sy,  cy, 0],
        [0,   0,  1],
    ])

    rx = np.array([
        [1, 0, 0],
        [0, cp, -sp],
        [0, sp,  cp],
    ])

    return points @ rz.T @ rx.T


def project(points, yaw, pitch):
    p = rotate(points, yaw, pitch)

    z = p[:, 2] + 12.0
    scale = 420 / z

    x2 = WIDTH * 0.5 + p[:, 0] * scale
    y2 = HEIGHT * 0.53 - p[:, 1] * scale

    return np.column_stack([x2, y2, z])


# ============================================================
# STARFIELD
# ============================================================

rng = np.random.default_rng(RNG_SEED)

stars = []
for _ in range(STAR_COUNT):
    stars.append((
        rng.uniform(0, WIDTH),
        rng.uniform(0, HEIGHT),
        rng.uniform(0.5, 1.7),
        rng.uniform(40, 120),
    ))


def draw_starfield(base, phase):
    d = ImageDraw.Draw(base)

    for i, (x, y, r, a) in enumerate(stars):
        twinkle = 0.65 + 0.35 * np.sin(
            phase * 2 * np.pi * rng.uniform(0.3, 1.7) + i
        )

        alpha = int(a * twinkle)

        d.ellipse(
            [x - r, y - r, x + r, y + r],
            fill=(190, 220, 255, alpha),
        )


# ============================================================
# ACCRETION DISK
# ============================================================

def draw_disk(base, phase):
    layer = rgba_layer()
    d = ImageDraw.Draw(layer)

    yaw = 2 * np.pi * CAMERA_ORBIT_SPEED * phase
    pitch = CAMERA_ELEVATION

    t = np.linspace(0, 2 * np.pi, 1200)

    rings = 42

    for k in range(rings):
        f = k / (rings - 1)

        r = (
            DISK_R_INNER
            + (DISK_R_OUTER - DISK_R_INNER) * f
        )

        turbulence = (
            1
            + 0.04 * np.sin(7 * t + phase * 12 + k * 0.2)
            + 0.03 * np.sin(13 * t - phase * 9)
        )

        rr = r * turbulence

        spin = phase * 2 * np.pi * (0.35 + 1.9 / (1 + f * 5))

        x = rr * np.cos(t + spin)
        y = rr * np.sin(t + spin)
        z = np.zeros_like(x)

        pts3 = np.column_stack([x, y, z]) * 0.012

        pts2 = project(pts3, yaw, pitch)

        pts = [(px, py) for px, py, _ in pts2]

        heat = 1.0 - f

        col = (
            int(255),
            int(180 + 70 * heat),
            int(90 + 160 * heat),
            int(16 + 46 * heat),
        )

        d.line(
            pts,
            fill=col,
            width=max(1, int(2.4 - 1.5 * f)),
            joint="curve",
        )

    add_glow(base, layer, blur=8)


# ============================================================
# BLACK HOLE
# ============================================================

def draw_black_hole(base, phase):
    layer = rgba_layer()
    d = ImageDraw.Draw(layer)

    cx = WIDTH * 0.5
    cy = HEIGHT * 0.53

    for rr, alpha in [
        (110, 20),
        (72, 42),
        (48, 90),
    ]:
        d.ellipse(
            [cx - rr, cy - rr, cx + rr, cy + rr],
            fill=(120, 220, 255, alpha),
        )

    d.ellipse(
        [cx - BH_RADIUS, cy - BH_RADIUS,
         cx + BH_RADIUS, cy + BH_RADIUS],
        fill=(0, 0, 0, 255),
    )

    d.ellipse(
        [cx - BH_RADIUS * 1.22, cy - BH_RADIUS * 0.92,
         cx + BH_RADIUS * 1.22, cy + BH_RADIUS * 0.92],
        outline=(220, 250, 255, 220),
        width=2,
    )

    add_glow(base, layer, blur=6)


# ============================================================
# JETS
# ============================================================

particles = []

for _ in range(PARTICLE_COUNT):
    particles.append((
        rng.uniform(-1, 1),
        rng.uniform(0, 1),
        rng.uniform(0.2, 1.0),
        rng.uniform(0.5, 1.5),
    ))


def draw_jets(base, phase):
    layer = rgba_layer()
    d = ImageDraw.Draw(layer)

    cx = WIDTH * 0.5
    cy = HEIGHT * 0.53

    for direction in [-1, 1]:

        yy, xx = np.mgrid[0:HEIGHT, 0:WIDTH]

        dx = xx - cx
        dy = (yy - cy) * direction

        forward = -dy
        side = dx

        cone = JET_WIDTH + forward * 0.09

        mask = (forward > 0) & (forward < JET_LENGTH)

        radial = np.exp(-(side ** 2) / (2 * cone ** 2))
        fade = np.clip(1 - forward / JET_LENGTH, 0, 1) ** 1.6

        alpha = radial * fade * mask

        arr = np.zeros((HEIGHT, WIDTH, 4), dtype=np.uint8)

        arr[..., 0] = 90
        arr[..., 1] = 220
        arr[..., 2] = 255
        arr[..., 3] = np.clip(alpha * 170, 0, 255).astype(np.uint8)

        jet = Image.fromarray(arr, "RGBA").filter(
            ImageFilter.GaussianBlur(8)
        )

        layer.alpha_composite(jet)

    # --------------------------------------------------------
    # particles inside jets
    # --------------------------------------------------------

    for px, speed, bright, size in particles:

        travel = (phase * speed * 2.4) % 1.0

        for direction in [-1, 1]:

            x = cx + px * (
                8 + travel * JET_WIDTH * 2.2
            )

            y = cy - direction * travel * JET_LENGTH

            alpha = int(120 * bright * (1.0 - travel))

            rr = size * (
                1.0 + 0.5 * np.sin(phase * 2 * np.pi * 4)
            )

            d.ellipse(
                [x - rr, y - rr, x + rr, y + rr],
                fill=(200, 255, 255, alpha),
            )

    add_glow(base, layer, blur=10)


# ============================================================
# SYNCHROTRON HELIX
# ============================================================

def draw_magnetic_helix(base, phase):
    layer = rgba_layer()
    d = ImageDraw.Draw(layer)

    yaw = 2 * np.pi * CAMERA_ORBIT_SPEED * phase
    pitch = CAMERA_ELEVATION

    for direction in [-1, 1]:

        t = np.linspace(0, 1, 1000)

        r = 0.12 + 0.26 * t

        theta = (
            24 * t
            + phase * 2 * np.pi * 1.8
        )

        x = r * np.cos(theta)
        y = direction * (0.2 + 8.0 * t)
        z = r * np.sin(theta)

        pts3 = np.column_stack([x, y, z])

        pts2 = project(pts3, yaw, pitch)

        pts = [(px, py) for px, py, _ in pts2]

        d.line(
            pts,
            fill=(170, 255, 255, 110),
            width=2,
            joint="curve",
        )

    add_glow(base, layer, blur=5)


# ============================================================
# FRAME
# ============================================================

def render_frame(i):
    phase = i / TOTAL_FRAMES

    base = Image.new("RGBA", (WIDTH, HEIGHT), BG)

    draw_starfield(base, phase)
    draw_disk(base, phase)
    draw_jets(base, phase)
    draw_magnetic_helix(base, phase)
    draw_black_hole(base, phase)

    return np.array(base.convert("RGB"))


# ============================================================
# RENDER
# ============================================================

frames = []

print("[START] relativistic jet")

for i in range(TOTAL_FRAMES):
    if i % FPS == 0:
        print(f"[RENDER] frame {i}/{TOTAL_FRAMES}")

    frames.append(render_frame(i))

if OUTPUT_FORMAT == "gif":
    imageio.mimsave(
        OUT_FILE,
        frames,
        fps=FPS,
        loop=0,
    )
else:
    imageio.mimsave(
        OUT_FILE,
        frames,
        fps=FPS,
        quality=9,
        macro_block_size=1,
    )

print()
print(f"[SAVED] {OUT_FILE.resolve()}")

[START] relativistic jet
[RENDER] frame 0/192


/var/folders/_b/cfj7mly10r9f3nkywrlqnl300000gn/T/ipykernel_1765/1392014087.py:276: DeprecationWarning: 'mode' parameter is deprecated and will be removed in Pillow 13 (2026-10-15)
  jet = Image.fromarray(arr, "RGBA").filter(


[RENDER] frame 24/192
[RENDER] frame 48/192
[RENDER] frame 72/192
[RENDER] frame 96/192
[RENDER] frame 120/192
[RENDER] frame 144/192
[RENDER] frame 168/192

[SAVED] /Users/mloktionov/PycharmProjects/Stellar_Attractor/ANIM/Infographics/Telemetry/animations/relativistic_jet/relativistic_jet.gif


This animation depicts a stylized active galactic nucleus powered by a supermassive black hole surrounded by a luminous accretion disk. Hot plasma spirals inward at relativistic speeds while twin polar jets erupt from the central region, carrying energetic particles thousands of light-years into intergalactic space. The glowing helical structures represent magnetic field lines guiding synchrotron-emitting plasma along the jets, inspired by observations of quasars, blazars, and radio galaxies captured by modern space observatories.

# Gravitational lensing / Einstein ring

In [ ]:
from __future__ import annotations

from pathlib import Path
import numpy as np
import imageio.v2 as imageio
from PIL import Image, ImageDraw, ImageFilter


# ============================================================
# Einstein Ring / Gravitational Lensing
# Stylized cinematic visualization
# ============================================================

OUTPUT_FORMAT = "gif"  # gif | mp4
FPS = 24
DURATION = 8
TOTAL_FRAMES = FPS * DURATION

WIDTH = 960
HEIGHT = 540

ANIMATION_NAME = "einstein_ring"
OUT_DIR = Path("media-site/animations") / ANIMATION_NAME
OUT_DIR.mkdir(parents=True, exist_ok=True)

OUT_FILE = OUT_DIR / f"{ANIMATION_NAME}.{OUTPUT_FORMAT}"

BG = (0, 0, 0, 255)

RNG_SEED = 42
STAR_COUNT = 1600

LENS_RADIUS = 54
RING_RADIUS = 118

DISTORTION_STRENGTH = 9800

CAMERA_ORBIT_SPEED = 0.16

ARC_COUNT = 9


# ============================================================
# HELPERS
# ============================================================

def rgba_layer():
    return Image.new("RGBA", (WIDTH, HEIGHT), (0, 0, 0, 0))


def add_glow(base, layer, blur=6):
    glow = layer.filter(ImageFilter.GaussianBlur(blur))
    base.alpha_composite(glow)
    base.alpha_composite(layer)


# ============================================================
# STARFIELD
# ============================================================

rng = np.random.default_rng(RNG_SEED)

stars = []

for _ in range(STAR_COUNT):
    stars.append((
        rng.uniform(0, WIDTH),
        rng.uniform(0, HEIGHT),
        rng.uniform(0.4, 2.2),
        rng.uniform(40, 180),
        rng.uniform(0.6, 1.0),
    ))


# ============================================================
# LENS DISTORTION
# ============================================================

def lens_point(x, y, phase):
    cx = WIDTH * 0.5
    cy = HEIGHT * 0.5

    dx = x - cx
    dy = y - cy

    r2 = dx * dx + dy * dy
    r = np.sqrt(r2) + 1e-6

    orbit = phase * 2 * np.pi * CAMERA_ORBIT_SPEED

    swirl = 0.06 * np.sin(orbit)

    angle = np.arctan2(dy, dx) + swirl / (1 + r * 0.01)

    bend = DISTORTION_STRENGTH / (r2 + 300)

    nx = x + np.cos(angle) * bend
    ny = y + np.sin(angle) * bend

    return nx, ny, r


# ============================================================
# BACKGROUND GALAXY FIELD
# ============================================================

def draw_background(base, phase):
    layer = rgba_layer()
    d = ImageDraw.Draw(layer)

    for i, (x0, y0, size, alpha, tint) in enumerate(stars):

        twinkle = (
            0.65
            + 0.35 * np.sin(
                phase * 2 * np.pi * rng.uniform(0.3, 1.5) + i
            )
        )

        x, y, r = lens_point(x0, y0, phase)

        if r < LENS_RADIUS:
            continue

        stretch = 1.0 + 2200 / (r * r + 200)

        sx = size * stretch
        sy = size / stretch

        col = (
            int(170 + 70 * tint),
            int(200 + 40 * tint),
            255,
            int(alpha * twinkle),
        )

        d.ellipse(
            [x - sx, y - sy, x + sx, y + sy],
            fill=col,
        )

    add_glow(base, layer, blur=2)


# ============================================================
# EINSTEIN RING
# ============================================================

def draw_ring(base, phase):
    layer = rgba_layer()
    d = ImageDraw.Draw(layer)

    cx = WIDTH * 0.5
    cy = HEIGHT * 0.5

    t = np.linspace(0, 2 * np.pi, 1600)

    pulse = 0.5 + 0.5 * np.sin(
        phase * 2 * np.pi * 2.0
    )

    ring_r = RING_RADIUS * (
        1.0
        + 0.02 * np.sin(6 * t + phase * 2 * np.pi * 3)
    )

    x = cx + ring_r * np.cos(t)
    y = cy + ring_r * np.sin(t)

    pts = list(zip(x, y))

    d.line(
        pts,
        fill=(210, 250, 255, int(210 + 30 * pulse)),
        width=3,
        joint="curve",
    )

    # outer glow
    for scale, alpha in [
        (1.04, 42),
        (1.08, 22),
    ]:
        rr = ring_r * scale
        xx = cx + rr * np.cos(t)
        yy = cy + rr * np.sin(t)

        d.line(
            list(zip(xx, yy)),
            fill=(120, 220, 255, alpha),
            width=2,
            joint="curve",
        )

    add_glow(base, layer, blur=6)


# ============================================================
# LENSED ARCS
# ============================================================

def draw_arcs(base, phase):
    layer = rgba_layer()
    d = ImageDraw.Draw(layer)

    cx = WIDTH * 0.5
    cy = HEIGHT * 0.5

    for k in range(ARC_COUNT):

        start = (
            2 * np.pi * k / ARC_COUNT
            + phase * 0.4
        )

        span = 0.5 + 0.6 * np.sin(k * 1.7)

        t = np.linspace(start, start + span, 420)

        rr = (
            RING_RADIUS
            + 22 * np.sin(phase * 2 * np.pi + k)
        )

        noise = (
            1
            + 0.03 * np.sin(11 * t + k)
            + 0.02 * np.sin(17 * t - phase * 4)
        )

        r = rr * noise

        x = cx + r * np.cos(t)
        y = cy + r * np.sin(t)

        alpha = int(
            120
            + 80 * (
                0.5
                + 0.5 * np.sin(phase * 2 * np.pi * 3 + k)
            )
        )

        d.line(
            list(zip(x, y)),
            fill=(160, 240, 255, alpha),
            width=2,
            joint="curve",
        )

    add_glow(base, layer, blur=5)


# ============================================================
# CENTRAL LENS OBJECT
# ============================================================

def draw_lens_mass(base, phase):
    layer = rgba_layer()
    d = ImageDraw.Draw(layer)

    cx = WIDTH * 0.5
    cy = HEIGHT * 0.5

    pulse = 0.5 + 0.5 * np.sin(
        phase * 2 * np.pi * 1.5
    ) ** 2

    for rr, alpha in [
        (110, 18),
        (72, 42),
        (48, 82),
    ]:
        rr *= 0.96 + 0.04 * pulse

        d.ellipse(
            [cx - rr, cy - rr,
             cx + rr, cy + rr],
            fill=(100, 210, 255, alpha),
        )

    d.ellipse(
        [cx - LENS_RADIUS,
         cy - LENS_RADIUS,
         cx + LENS_RADIUS,
         cy + LENS_RADIUS],
        fill=(0, 0, 0, 255),
    )

    d.ellipse(
        [cx - LENS_RADIUS * 1.18,
         cy - LENS_RADIUS * 1.18,
         cx + LENS_RADIUS * 1.18,
         cy + LENS_RADIUS * 1.18],
        outline=(220, 250, 255, 230),
        width=2,
    )

    add_glow(base, layer, blur=8)


# ============================================================
# CAUSTIC SHIMMER
# ============================================================

def draw_caustics(base, phase):
    layer = rgba_layer()
    d = ImageDraw.Draw(layer)

    cx = WIDTH * 0.5
    cy = HEIGHT * 0.5

    for i in range(18):

        angle = (
            2 * np.pi * i / 18
            + phase * 2 * np.pi * 0.12
        )

        r0 = RING_RADIUS * 0.75
        r1 = RING_RADIUS * 1.7

        x0 = cx + r0 * np.cos(angle)
        y0 = cy + r0 * np.sin(angle)

        x1 = cx + r1 * np.cos(angle)
        y1 = cy + r1 * np.sin(angle)

        alpha = int(
            10
            + 16 * (
                0.5
                + 0.5 * np.sin(phase * 2 * np.pi * 2 + i)
            )
        )

        d.line(
            [(x0, y0), (x1, y1)],
            fill=(120, 220, 255, alpha),
            width=1,
        )

    add_glow(base, layer, blur=12)


# ============================================================
# FRAME
# ============================================================

def render_frame(i):
    phase = i / TOTAL_FRAMES

    base = Image.new("RGBA", (WIDTH, HEIGHT), BG)

    draw_background(base, phase)
    
    draw_caustics(base, phase)
    draw_arcs(base, phase)
    draw_ring(base, phase)
    draw_lens_mass(base, phase)

    return np.array(base.convert("RGB"))


# ============================================================
# RENDER
# ============================================================

frames = []

print("[START] Einstein Ring")

for i in range(TOTAL_FRAMES):

    if i % FPS == 0:
        print(f"[RENDER] frame {i}/{TOTAL_FRAMES}")

    frames.append(render_frame(i))

if OUTPUT_FORMAT == "gif":
    imageio.mimsave(
        OUT_FILE,
        frames,
        fps=FPS,
        loop=0,
    )
else:
    imageio.mimsave(
        OUT_FILE,
        frames,
        fps=FPS,
        quality=9,
        macro_block_size=1,
    )

print()
print(f"[SAVED] {OUT_FILE.resolve()}")

[START] Einstein Ring
[RENDER] frame 0/192
[RENDER] frame 24/192
[RENDER] frame 48/192
[RENDER] frame 72/192
[RENDER] frame 96/192
[RENDER] frame 120/192
[RENDER] frame 144/192
[RENDER] frame 168/192

[SAVED] /Users/mloktionov/PycharmProjects/Stellar_Attractor/ANIM/Infographics/Telemetry/animations/einstein_ring/einstein_ring.gif


This animation illustrates gravitational lensing caused by an extremely massive compact object bending the paths of light from distant background stars and galaxies. As spacetime curves around the central mass, light is warped into luminous arcs and a nearly complete Einstein ring — one of the most striking predictions of General Relativity. The shimmering distortions and caustic structures mimic the optical effects observed in deep-field astronomical surveys, where foreground galaxies and dark matter halos magnify and reshape the appearance of objects located billions of light-years behind them.

# Cosmic web flythrough

In [ ]:
from __future__ import annotations

from pathlib import Path
import numpy as np
import imageio.v2 as imageio
from PIL import Image, ImageDraw, ImageFilter


# ============================================================
# Cosmic Web Flythrough
# Stylized large-scale structure visualization
# ============================================================

OUTPUT_FORMAT = "gif"  # gif | mp4
FPS = 24
DURATION = 8
TOTAL_FRAMES = FPS * DURATION

WIDTH = 960
HEIGHT = 540

ANIMATION_NAME = "cosmic_web_flythrough"

OUT_DIR = Path("media-site/animations") / ANIMATION_NAME
OUT_DIR.mkdir(parents=True, exist_ok=True)

OUT_FILE = OUT_DIR / f"{ANIMATION_NAME}.{OUTPUT_FORMAT}"

BG = (0, 0, 0, 255)

RNG_SEED = 42

NODE_COUNT = 55
FILAMENT_POINTS = 120
PARTICLE_COUNT = 2200

CAMERA_SPEED = 0.028

FOG_LAYERS = 18


# ============================================================
# HELPERS
# ============================================================

def rgba():
    return Image.new("RGBA", (WIDTH, HEIGHT), (0, 0, 0, 0))


def add_glow(base, layer, blur=6):
    glow = layer.filter(ImageFilter.GaussianBlur(blur))
    base.alpha_composite(glow)
    base.alpha_composite(layer)


rng = np.random.default_rng(RNG_SEED)


# ============================================================
# 3D COSMIC WEB NODES
# ============================================================

nodes = []

for _ in range(NODE_COUNT):

    x = rng.uniform(-14, 14)
    y = rng.uniform(-8, 8)
    z = rng.uniform(2, 34)

    brightness = rng.uniform(0.5, 1.0)

    nodes.append(np.array([x, y, z, brightness]))


# ============================================================
# CONNECT FILAMENTS
# ============================================================

filaments = []

for i, n1 in enumerate(nodes):

    pos1 = n1[:3]

    dists = []

    for j, n2 in enumerate(nodes):

        if i == j:
            continue

        pos2 = n2[:3]

        d = np.linalg.norm(pos1 - pos2)

        dists.append((d, j))

    dists.sort(key=lambda x: x[0])

    for _, idx in dists[:3]:

        if idx <= i:
            continue

        p1 = nodes[i][:3]
        p2 = nodes[idx][:3]

        pts = []

        for t in np.linspace(0, 1, FILAMENT_POINTS):

            curve = (
                0.45
                * np.sin(t * np.pi)
                * np.array([
                    np.sin(i * 0.7),
                    np.cos(idx * 0.5),
                    0
                ])
            )

            p = p1 * (1 - t) + p2 * t + curve

            pts.append(p)

        filaments.append(np.array(pts))


# ============================================================
# PARTICLES INSIDE FILAMENTS
# ============================================================

particles = []

for _ in range(PARTICLE_COUNT):

    filament = filaments[rng.integers(0, len(filaments))]

    t = rng.uniform(0, 1)

    idx = int(t * (len(filament) - 1))

    p = filament[idx].copy()

    jitter = rng.normal(0, 0.12, size=3)

    p += jitter

    speed = rng.uniform(0.2, 1.2)

    particles.append([p, speed])


# ============================================================
# PROJECTION
# ============================================================

def project(points, camera_z):

    pts = points.copy()

    pts[:, 2] -= camera_z

    visible = pts[:, 2] > 0.1

    pts = pts[visible]

    if len(pts) == 0:
        return np.zeros((0, 3))

    z = pts[:, 2]

    scale = 520 / z

    x = WIDTH * 0.5 + pts[:, 0] * scale
    y = HEIGHT * 0.5 - pts[:, 1] * scale

    return np.column_stack([x, y, z])


# ============================================================
# STAR BACKGROUND
# ============================================================

bg_stars = []

for _ in range(1000):

    bg_stars.append((
        rng.uniform(0, WIDTH),
        rng.uniform(0, HEIGHT),
        rng.uniform(0.4, 1.7),
        rng.uniform(15, 80),
    ))


def draw_background(base, phase):

    d = ImageDraw.Draw(base)

    for i, (x, y, r, a) in enumerate(bg_stars):

        tw = 0.7 + 0.3 * np.sin(phase * 6 + i)

        d.ellipse(
            [x-r, y-r, x+r, y+r],
            fill=(180, 220, 255, int(a * tw)),
        )


# ============================================================
# FOG / GAS
# ============================================================

def draw_fog(base, phase, camera_z):

    layer = rgba()
    d = ImageDraw.Draw(layer)

    for i in range(FOG_LAYERS):

        z = (
            i / FOG_LAYERS * 28
            + phase * 8
        ) % 28

        scale = 1.0 + 6 / (z + 1)

        alpha = int(8 + 12 / (z + 1))

        cx = WIDTH * 0.5 + np.sin(i * 1.3 + phase * 2) * 180
        cy = HEIGHT * 0.5 + np.cos(i * 1.7 + phase * 1.5) * 90

        rr = 160 * scale

        d.ellipse(
            [cx-rr, cy-rr, cx+rr, cy+rr],
            fill=(70, 140, 255, alpha),
        )

    add_glow(base, layer, blur=32)


# ============================================================
# FILAMENTS
# ============================================================

def draw_filaments(base, phase, camera_z):

    layer = rgba()
    d = ImageDraw.Draw(layer)

    for filament in filaments:

        pts = project(filament, camera_z)

        if len(pts) < 2:
            continue

        poly = [(x, y) for x, y, _ in pts]

        depth = np.mean(pts[:, 2])

        alpha = int(np.clip(180 - depth * 4, 12, 110))

        d.line(
            poly,
            fill=(110, 220, 255, alpha),
            width=2,
            joint="curve",
        )

    add_glow(base, layer, blur=7)


# ============================================================
# GALAXY CLUSTERS
# ============================================================

def draw_nodes(base, phase, camera_z):

    layer = rgba()
    d = ImageDraw.Draw(layer)

    pts = project(np.array([n[:3] for n in nodes]), camera_z)

    if len(pts) == 0:
        return

    for i, (x, y, z) in enumerate(pts):

        brightness = nodes[i][3]

        pulse = (
            0.65
            + 0.35 * np.sin(
                phase * 2 * np.pi * brightness * 2
                + i
            )
        )

        rr = np.clip(22 / z, 1.5, 9)

        for scale, alpha in [
            (6.0, 12),
            (3.0, 32),
            (1.6, 80),
        ]:

            r2 = rr * scale

            d.ellipse(
                [x-r2, y-r2, x+r2, y+r2],
                fill=(
                    170,
                    240,
                    255,
                    int(alpha * pulse),
                ),
            )

    add_glow(base, layer, blur=12)


# ============================================================
# PARTICLE STREAMS
# ============================================================

def draw_particles(base, phase, camera_z):

    layer = rgba()
    d = ImageDraw.Draw(layer)

    for p, speed in particles:

        pos = p.copy()

        pos[2] -= phase * speed * 12

        pts = project(np.array([pos]), camera_z)

        if len(pts) == 0:
            continue

        x, y, z = pts[0]

        rr = np.clip(5 / z, 0.5, 2.2)

        alpha = int(np.clip(180 - z * 5, 18, 140))

        d.ellipse(
            [x-rr, y-rr, x+rr, y+rr],
            fill=(220, 255, 255, alpha),
        )

    add_glow(base, layer, blur=4)


# ============================================================
# MAIN FRAME
# ============================================================

def render_frame(i):

    phase = i / TOTAL_FRAMES

    camera_z = phase * CAMERA_SPEED * 220

    base = Image.new("RGBA", (WIDTH, HEIGHT), BG)

    draw_background(base, phase)
    draw_fog(base, phase, camera_z)
    draw_filaments(base, phase, camera_z)
    draw_particles(base, phase, camera_z)
    draw_nodes(base, phase, camera_z)

    return np.array(base.convert("RGB"))


# ============================================================
# RENDER
# ============================================================

frames = []

print("[START] Cosmic Web Flythrough")

for i in range(TOTAL_FRAMES):

    if i % FPS == 0:
        print(f"[RENDER] frame {i}/{TOTAL_FRAMES}")

    frames.append(render_frame(i))

if OUTPUT_FORMAT == "gif":

    imageio.mimsave(
        OUT_FILE,
        frames,
        fps=FPS,
        loop=0,
    )

else:

    imageio.mimsave(
        OUT_FILE,
        frames,
        fps=FPS,
        quality=9,
        macro_block_size=1,
    )

print()
print(f"[SAVED] {OUT_FILE.resolve()}")

[START] Cosmic Web Flythrough
[RENDER] frame 0/192
[RENDER] frame 24/192
[RENDER] frame 48/192
[RENDER] frame 72/192
[RENDER] frame 96/192
[RENDER] frame 120/192
[RENDER] frame 144/192
[RENDER] frame 168/192

[SAVED] /Users/mloktionov/PycharmProjects/Stellar_Attractor/ANIM/Infographics/Telemetry/animations/cosmic_web_flythrough/cosmic_web_flythrough.gif


This animation represents the large-scale structure of the Universe — the so-called Cosmic Web — formed by vast filaments of dark matter, diffuse gas, and galaxy clusters stretching across hundreds of millions of light-years. Matter is not distributed uniformly through space: gravity gradually pulls galaxies and intergalactic plasma into enormous interconnected threads, while immense cosmic voids remain nearly empty between them. The glowing nodes correspond to dense galaxy clusters located at the intersections of filaments, while the flowing luminous streams evoke the motion of baryonic gas and matter accreting along the underlying dark matter skeleton of the Universe.

# Relativistic accretion disk / photon orbit visualization

In [ ]:
from __future__ import annotations

from pathlib import Path
import numpy as np
import imageio.v2 as imageio
from PIL import Image, ImageDraw, ImageFilter


# ============================================================
# Relativistic Accretion Disk
# Cinematic black hole visualization
# ============================================================

OUTPUT_FORMAT = "gif"   # gif | mp4
FPS = 24
DURATION = 8
TOTAL_FRAMES = FPS * DURATION

WIDTH = 960
HEIGHT = 540

ANIMATION_NAME = "relativistic_accretion_disk"

OUT_DIR = Path("media-site/animations") / ANIMATION_NAME
OUT_DIR.mkdir(parents=True, exist_ok=True)

OUT_FILE = OUT_DIR / f"{ANIMATION_NAME}.{OUTPUT_FORMAT}"

BG = (0, 0, 0, 255)

RNG_SEED = 42

BH_RADIUS = 52

DISK_INNER = 92
DISK_OUTER = 260

CAMERA_TILT = np.deg2rad(72)

HOTSPOT_COUNT = 8

STAR_COUNT = 1200

FRAME_DRAGGING = 0.65

DOPPLER_STRENGTH = 1.8

PHOTON_RING_RADIUS = 78


# ============================================================
# HELPERS
# ============================================================

def rgba():
    return Image.new("RGBA", (WIDTH, HEIGHT), (0, 0, 0, 0))


def add_glow(base, layer, blur=6):
    glow = layer.filter(ImageFilter.GaussianBlur(blur))
    base.alpha_composite(glow)
    base.alpha_composite(layer)


rng = np.random.default_rng(RNG_SEED)


# ============================================================
# STARFIELD
# ============================================================

stars = []

for _ in range(STAR_COUNT):

    stars.append((
        rng.uniform(0, WIDTH),
        rng.uniform(0, HEIGHT),
        rng.uniform(0.3, 1.8),
        rng.uniform(20, 140),
    ))


def draw_background(base, phase):

    d = ImageDraw.Draw(base)

    for i, (x, y, r, a) in enumerate(stars):

        twinkle = (
            0.72
            + 0.28 * np.sin(i * 0.7 + phase * 8)
        )

        d.ellipse(
            [x-r, y-r, x+r, y+r],
            fill=(180, 220, 255, int(a * twinkle)),
        )

def apply_solar_disk_mask(layer, cx, cy, radius=SUN_RADIUS):
    arr = np.array(layer)

    yy, xx = np.mgrid[0:HEIGHT, 0:WIDTH]
    r = np.sqrt((xx - cx) ** 2 + (yy - cy) ** 2)

    mask = r <= radius * 0.99
    arr[..., 3] = np.where(mask, 0, arr[..., 3])

    return Image.fromarray(arr.astype(np.uint8), "RGBA")


def add_glow_occluded(base, layer, cx, cy, blur=8):
    masked_layer = apply_solar_disk_mask(layer, cx, cy)
    glow = masked_layer.filter(ImageFilter.GaussianBlur(blur))
    glow = apply_solar_disk_mask(glow, cx, cy)

    base.alpha_composite(glow)
    base.alpha_composite(masked_layer)
    
# ============================================================
# DISK PROJECTION
# ============================================================

def project_disk(r, theta, phase):

    spin = (
        theta
        + phase * 2 * np.pi * (1.2 + FRAME_DRAGGING / (r * 0.01))
    )

    x = r * np.cos(spin)
    y = r * np.sin(spin)

    yy = y * np.cos(CAMERA_TILT)

    cx = WIDTH * 0.5
    cy = HEIGHT * 0.5

    return cx + x, cy + yy


# ============================================================
# RELATIVISTIC DISK
# ============================================================

def draw_disk(base, phase):

    layer = rgba()
    d = ImageDraw.Draw(layer)

    rings = 240

    for i in range(rings):

        frac = i / (rings - 1)

        r = DISK_INNER + frac * (DISK_OUTER - DISK_INNER)

        segments = int(220 + r * 0.9)

        prev = None

        for j in range(segments + 1):

            theta = 2 * np.pi * j / segments

            x, y = project_disk(r, theta, phase)

            # ------------------------------------------------
            # Doppler boosting
            # ------------------------------------------------

            vel = np.sin(theta + phase * 2 * np.pi * 1.3)

            doppler = (
                0.45
                + DOPPLER_STRENGTH * max(0, vel)
            )

            redshift = (
                0.35
                + 0.65 * max(0, -vel)
            )

            heat = 1.0 - frac

            rr = int(255 * min(1.0, 0.7 * doppler + 0.3 * heat))
            gg = int(160 + 80 * heat)
            bb = int(80 + 190 * redshift)

            alpha = int(
                np.clip(
                    20 + 90 * heat * doppler,
                    8,
                    190
                )
            )

            pt = (x, y)

            if prev is not None:

                d.line(
                    [prev, pt],
                    fill=(rr, gg, bb, alpha),
                    width=2,
                    joint="curve",
                )

            prev = pt

    add_glow(base, layer, blur=7)


# ============================================================
# PHOTON RING
# ============================================================

def draw_photon_ring(base, phase):

    layer = rgba()
    d = ImageDraw.Draw(layer)

    cx = WIDTH * 0.5
    cy = HEIGHT * 0.5

    pulse = (
        0.75
        + 0.25 * np.sin(phase * 2 * np.pi * 3)
    )

    for scale, alpha in [
        (1.0, 180),
        (1.12, 60),
        (1.25, 20),
    ]:

        rr = PHOTON_RING_RADIUS * scale

        d.ellipse(
            [
                cx - rr,
                cy - rr * 0.74,
                cx + rr,
                cy + rr * 0.74,
            ],
            outline=(
                220,
                250,
                255,
                int(alpha * pulse),
            ),
            width=2,
        )

    add_glow(base, layer, blur=6)


# ============================================================
# HOTSPOTS
# ============================================================

def draw_hotspots(base, phase):

    layer = rgba()
    d = ImageDraw.Draw(layer)

    for i in range(HOTSPOT_COUNT):

        t = (
            phase * 2 * np.pi * (1.8 + i * 0.1)
            + i * 2 * np.pi / HOTSPOT_COUNT
        )

        r = (
            DISK_INNER
            + 28
            + i * 14
        )

        x, y = project_disk(r, t, phase)

        pulse = (
            0.55
            + 0.45 * np.sin(t * 2)
        )

        rr = 5 + 7 * pulse

        for scale, alpha in [
            (3.0, 18),
            (1.8, 60),
        ]:

            r2 = rr * scale

            d.ellipse(
                [x-r2, y-r2, x+r2, y+r2],
                fill=(
                    255,
                    240,
                    170,
                    int(alpha * pulse),
                ),
            )

        d.ellipse(
            [x-rr, y-rr, x+rr, y+rr],
            fill=(255, 250, 220, 220),
        )

    add_glow(base, layer, blur=8)


# ============================================================
# LENSED FAR SIDE
# ============================================================

def draw_lensed_backside(base, phase):

    layer = rgba()
    d = ImageDraw.Draw(layer)

    cx = WIDTH * 0.5
    cy = HEIGHT * 0.5

    t = np.linspace(0, 2 * np.pi, 1000)

    rr = DISK_INNER * 0.92

    x = cx + rr * np.cos(t)
    y = cy - rr * 0.30 * np.sin(t)

    pts = list(zip(x, y))

    d.line(
        pts,
        fill=(120, 210, 255, 90),
        width=3,
        joint="curve",
    )

    add_glow(base, layer, blur=8)


# ============================================================
# EVENT HORIZON
# ============================================================

def draw_black_hole(base):

    layer = rgba()
    d = ImageDraw.Draw(layer)

    cx = WIDTH * 0.5
    cy = HEIGHT * 0.5

    for rr, alpha in [
        (140, 10),
        (100, 22),
        (72, 46),
    ]:

        d.ellipse(
            [
                cx - rr,
                cy - rr,
                cx + rr,
                cy + rr,
            ],
            fill=(100, 210, 255, alpha),
        )

    d.ellipse(
        [
            cx - BH_RADIUS,
            cy - BH_RADIUS,
            cx + BH_RADIUS,
            cy + BH_RADIUS,
        ],
        fill=(0, 0, 0, 255),
    )

    add_glow(base, layer, blur=12)


# ============================================================
# RELATIVISTIC STREAKS
# ============================================================

def draw_streaks(base, phase):

    layer = rgba()
    d = ImageDraw.Draw(layer)

    cx = WIDTH * 0.5
    cy = HEIGHT * 0.5

    for i in range(24):

        angle = (
            i * 2 * np.pi / 24
            + phase * 0.2
        )

        r0 = DISK_OUTER * 0.75
        r1 = DISK_OUTER * 1.08

        x0 = cx + r0 * np.cos(angle)
        y0 = cy + r0 * np.sin(angle) * np.cos(CAMERA_TILT)

        x1 = cx + r1 * np.cos(angle)
        y1 = cy + r1 * np.sin(angle) * np.cos(CAMERA_TILT)

        alpha = int(
            10
            + 12 * (
                0.5
                + 0.5 * np.sin(phase * 5 + i)
            )
        )

        d.line(
            [(x0, y0), (x1, y1)],
            fill=(120, 220, 255, alpha),
            width=1,
        )

    add_glow(base, layer, blur=10)


# ============================================================
# FRAME
# ============================================================

def render_frame(i):

    phase = i / TOTAL_FRAMES

    base = Image.new("RGBA", (WIDTH, HEIGHT), BG)

    draw_background(base, phase)

    draw_streaks(base, phase)

    draw_disk(base, phase)

    draw_hotspots(base, phase)

    draw_lensed_backside(base, phase)

    draw_photon_ring(base, phase)

    draw_black_hole(base)

    return np.array(base.convert("RGB"))


# ============================================================
# RENDER
# ============================================================

frames = []

print("[START] Relativistic Accretion Disk")

for i in range(TOTAL_FRAMES):

    if i % FPS == 0:
        print(f"[RENDER] frame {i}/{TOTAL_FRAMES}")

    frames.append(render_frame(i))

if OUTPUT_FORMAT == "gif":

    imageio.mimsave(
        OUT_FILE,
        frames,
        fps=FPS,
        loop=0,
    )

else:

    imageio.mimsave(
        OUT_FILE,
        frames,
        fps=FPS,
        quality=9,
        macro_block_size=1,
    )

print()
print(f"[SAVED] {OUT_FILE.resolve()}")

[START] Relativistic Accretion Disk
[RENDER] frame 0/192
[RENDER] frame 24/192
[RENDER] frame 48/192
[RENDER] frame 72/192
[RENDER] frame 96/192
[RENDER] frame 120/192
[RENDER] frame 144/192
[RENDER] frame 168/192

[SAVED] /Users/mloktionov/PycharmProjects/Stellar_Attractor/ANIM/Infographics/Telemetry/animations/relativistic_accretion_disk/relativistic_accretion_disk.gif


This animation visualizes a relativistic accretion disk surrounding a rotating black hole. Superheated plasma orbits the event horizon at enormous fractions of the speed of light, producing intense radiation and dramatic relativistic effects predicted by General Relativity. The bright asymmetric side of the disk is enhanced by Doppler boosting as matter moves toward the observer, while the dimmer side is redshifted as it recedes. The luminous photon ring and distorted upper arc represent gravitational lensing of light from the far side of the disk, where spacetime curvature bends photon trajectories around the black hole before they escape toward the observer.

# Solar Coronal Loops / CME Eruption

In [31]:
from __future__ import annotations

from pathlib import Path
import numpy as np
import imageio.v2 as imageio
from PIL import Image, ImageDraw, ImageFilter


# ============================================================
# Solar Coronal Loops + CME Eruption
# Stylized heliophysics visualization
# ============================================================

OUTPUT_FORMAT = "gif"   # gif | mp4
FPS = 24
DURATION = 8
TOTAL_FRAMES = FPS * DURATION

WIDTH = 960
HEIGHT = 540

ANIMATION_NAME = "solar_coronal_loops"

OUT_DIR = Path("media-site/animations") / ANIMATION_NAME
OUT_DIR.mkdir(parents=True, exist_ok=True)

OUT_FILE = OUT_DIR / f"{ANIMATION_NAME}.{OUTPUT_FORMAT}"

BG = (0, 0, 0, 255)

RNG_SEED = 42

SUN_RADIUS = 150

LOOP_COUNT = 42
PLASMA_PARTICLES = 900

CME_PHASE = 0.62

CAMERA_DRIFT = 18


# ============================================================
# HELPERS
# ============================================================

def rgba():
    return Image.new("RGBA", (WIDTH, HEIGHT), (0, 0, 0, 0))


def add_glow(base, layer, blur=8):
    glow = layer.filter(ImageFilter.GaussianBlur(blur))
    base.alpha_composite(glow)
    base.alpha_composite(layer)


rng = np.random.default_rng(RNG_SEED)

def apply_solar_disk_mask(layer, cx, cy, radius=SUN_RADIUS):
    arr = np.array(layer)

    yy, xx = np.mgrid[0:HEIGHT, 0:WIDTH]
    r = np.sqrt((xx - cx) ** 2 + (yy - cy) ** 2)

    mask = r <= radius * 0.99
    arr[..., 3] = np.where(mask, 0, arr[..., 3])

    return Image.fromarray(arr.astype(np.uint8), "RGBA")


def add_glow_occluded(base, layer, cx, cy, blur=8):
    masked_layer = apply_solar_disk_mask(layer, cx, cy)
    glow = masked_layer.filter(ImageFilter.GaussianBlur(blur))
    glow = apply_solar_disk_mask(glow, cx, cy)

    base.alpha_composite(glow)
    base.alpha_composite(masked_layer)

# ============================================================
# BACKGROUND STARS
# ============================================================

stars = []

for _ in range(1200):

    stars.append((
        rng.uniform(0, WIDTH),
        rng.uniform(0, HEIGHT),
        rng.uniform(0.4, 1.7),
        rng.uniform(20, 120),
    ))


def draw_background(base, phase):

    d = ImageDraw.Draw(base)

    for i, (x, y, r, a) in enumerate(stars):

        twinkle = (
            0.72
            + 0.28 * np.sin(i * 0.3 + phase * 6)
        )

        d.ellipse(
            [x-r, y-r, x+r, y+r],
            fill=(180, 220, 255, int(a * twinkle)),
        )

def mask_background_behind_sun(base, cx, cy):

    arr = np.array(base)

    yy, xx = np.mgrid[0:HEIGHT, 0:WIDTH]

    r = np.sqrt(
        (xx - cx) ** 2 +
        (yy - cy) ** 2
    )

    # чуть больше фотосферы
    mask = r <= SUN_RADIUS * 1.02

    arr[mask, 0] = 0
    arr[mask, 1] = 0
    arr[mask, 2] = 0

    base.paste(Image.fromarray(arr.astype(np.uint8), "RGBA"))

# ============================================================
# SUN
# ============================================================

def draw_sun(base, phase):

    layer = rgba()
    d = ImageDraw.Draw(layer)

    cx = WIDTH * 0.42 + CAMERA_DRIFT * np.sin(phase * 2 * np.pi * 0.25)
    cy = HEIGHT * 0.52

    yy, xx = np.mgrid[0:HEIGHT, 0:WIDTH]

    dx = xx - cx
    dy = yy - cy

    r = np.sqrt(dx * dx + dy * dy)

    sphere = r <= SUN_RADIUS

    shade = np.clip(
        1.0 - r / SUN_RADIUS,
        0,
        1,
    )

    turbulence = (
        0.45
        + 0.25 * np.sin(dx * 0.04 + phase * 7)
        + 0.18 * np.sin(dy * 0.05 - phase * 9)
        + 0.14 * np.sin((dx + dy) * 0.03 + phase * 11)
    )

    turbulence = np.clip(turbulence, 0, 1)

    intensity = shade ** 0.42 * turbulence

    arr = np.zeros((HEIGHT, WIDTH, 4), dtype=np.uint8)

    arr[..., 0] = np.where(sphere, 255, 0)
    arr[..., 1] = np.where(sphere, 130 + 90 * intensity, 0)
    arr[..., 2] = np.where(sphere, 30 + 50 * intensity, 0)
    arr[..., 3] = np.where(sphere, 255, 0)

    img = Image.fromarray(arr.astype(np.uint8), "RGBA")

    layer.alpha_composite(img)

    # corona glow
    for rr, alpha in [
        (240, 10),
        (200, 20),
        (170, 42),
    ]:

        d.ellipse(
            [
                cx - rr,
                cy - rr,
                cx + rr,
                cy + rr,
            ],
            fill=(255, 160, 50, alpha),
        )

    add_glow(base, layer, blur=18)

    return cx, cy


# ============================================================
# MAGNETIC LOOPS
# ============================================================

loop_specs = []

for i in range(LOOP_COUNT):

    a0 = rng.uniform(-1.2, 1.2)
    span = rng.uniform(0.25, 1.1)

    height = rng.uniform(70, 220)

    thickness = rng.uniform(1, 3)

    side = rng.choice([-1, 1])

    loop_specs.append((
        a0,
        span,
        height,
        thickness,
        side,
    ))


def draw_loops(base, phase, cx, cy):

    layer = rgba()
    d = ImageDraw.Draw(layer)

    eruption_t = np.clip(
        (phase - CME_PHASE) / 0.18,
        0,
        1,
    )

    for i, (a0, span, height, thickness, side) in enumerate(loop_specs):

        t = np.linspace(0, 1, 240)

        angle0 = a0
        angle1 = a0 + side * span

        theta = angle0 * (1 - t) + angle1 * t

        surface_r = SUN_RADIUS * 0.96

        x0 = cx + surface_r * np.cos(theta)
        y0 = cy + surface_r * np.sin(theta)

        arc = np.sin(t * np.pi)

        loop_h = (
            height
            * (
                1
                + 0.08 * np.sin(phase * 2 * np.pi * 2 + i)
            )
        )

        # erupting loops
        if i < 6:
            loop_h *= 1.0 + 3.2 * eruption_t

        x = x0
        y = y0 - arc * loop_h

        pts = list(zip(x, y))

        pulse = (
            0.65
            + 0.35 * np.sin(
                phase * 2 * np.pi * 3
                + i
            )
        )

        alpha = int(55 + 90 * pulse)

        col = (
            120,
            240,
            255,
            alpha,
        )

        d.line(
            pts,
            fill=col,
            width=int(thickness),
            joint="curve",
        )

    add_glow_occluded(base, layer, cx, cy, blur=7)

# ============================================================
# PLASMA FLOW
# ============================================================

particles = []

for _ in range(PLASMA_PARTICLES):

    loop_id = rng.integers(0, LOOP_COUNT)

    t = rng.uniform(0, 1)

    speed = rng.uniform(0.3, 1.3)

    particles.append([loop_id, t, speed])


def draw_plasma(base, phase, cx, cy):

    layer = rgba()
    d = ImageDraw.Draw(layer)

    eruption_t = np.clip(
        (phase - CME_PHASE) / 0.18,
        0,
        1,
    )

    for loop_id, t0, speed in particles:

        a0, span, height, thickness, side = loop_specs[loop_id]

        t = (t0 + phase * speed) % 1.0

        theta = a0 * (1 - t) + (a0 + side * span) * t

        x0 = cx + SUN_RADIUS * 0.96 * np.cos(theta)
        y0 = cy + SUN_RADIUS * 0.96 * np.sin(theta)

        loop_h = height

        if loop_id < 6:
            loop_h *= 1.0 + 3.2 * eruption_t

        arc = np.sin(t * np.pi)

        x = x0
        y = y0 - arc * loop_h

        rr = 1.2 + 1.8 * arc

        alpha = int(
            90
            + 120 * arc
        )

        d.ellipse(
            [x-rr, y-rr, x+rr, y+rr],
            fill=(220, 255, 255, alpha),
        )

    add_glow_occluded(base, layer, cx, cy, blur=4)

# ============================================================
# CME ERUPTION
# ============================================================

def draw_cme(base, phase, cx, cy):

    eruption_t = np.clip(
        (phase - CME_PHASE) / 0.25,
        0,
        1,
    )

    if eruption_t <= 0:
        return

    layer = rgba()
    d = ImageDraw.Draw(layer)

    theta = np.linspace(-0.7, 0.7, 500)

    radius = (
        180
        + eruption_t * 520
    )

    deform = (
        1
        + 0.08 * np.sin(theta * 7 + phase * 8)
    )

    x = (
        cx
        + radius * np.sin(theta) * deform
    )

    y = (
        cy
        - radius * np.cos(theta) * deform
        - 170 * eruption_t
    )

    pts = list(zip(x, y))

    for width, alpha in [
        (22, 12),
        (14, 22),
        (7, 55),
    ]:

        d.line(
            pts,
            fill=(120, 240, 255, alpha),
            width=width,
            joint="curve",
        )

    add_glow_occluded(base, layer, cx, cy, blur=18)

# ============================================================
# SHOCKWAVE
# ============================================================

def draw_shockwave(base, phase, cx, cy):

    eruption_t = np.clip(
        (phase - CME_PHASE) / 0.20,
        0,
        1,
    )

    if eruption_t <= 0:
        return

    layer = rgba()
    d = ImageDraw.Draw(layer)

    rr = (
        120
        + eruption_t * 520
    )

    alpha = int(40 * (1 - eruption_t))

    d.ellipse(
        [
            cx - rr,
            cy - rr,
            cx + rr,
            cy + rr,
        ],
        outline=(160, 240, 255, alpha),
        width=3,
    )

    add_glow_occluded(base, layer, cx, cy, blur=12)


# ============================================================
# FRAME
# ============================================================
def render_frame(i):

    phase = i / TOTAL_FRAMES

    base = Image.new("RGBA", (WIDTH, HEIGHT), BG)

    draw_background(base, phase)

    cx, cy = draw_sun(base, phase)

    mask_background_behind_sun(base, cx, cy)

    draw_loops(base, phase, cx, cy)
    draw_plasma(base, phase, cx, cy)
    draw_shockwave(base, phase, cx, cy)
    draw_cme(base, phase, cx, cy)

    draw_solar_occlusion(base, phase, cx, cy)

    return np.array(base.convert("RGB"))


# ============================================================
# RENDER
# ============================================================

frames = []

print("[START] Solar Coronal Loops")

for i in range(TOTAL_FRAMES):

    if i % FPS == 0:
        print(f"[RENDER] frame {i}/{TOTAL_FRAMES}")

    frames.append(render_frame(i))

if OUTPUT_FORMAT == "gif":

    imageio.mimsave(
        OUT_FILE,
        frames,
        fps=FPS,
        loop=0,
    )

else:

    imageio.mimsave(
        OUT_FILE,
        frames,
        fps=FPS,
        quality=9,
        macro_block_size=1,
    )

print()
print(f"[SAVED] {OUT_FILE.resolve()}")

[START] Solar Coronal Loops
[RENDER] frame 0/192


/var/folders/_b/cfj7mly10r9f3nkywrlqnl300000gn/T/ipykernel_1765/1337777759.py:176: DeprecationWarning: 'mode' parameter is deprecated and will be removed in Pillow 13 (2026-10-15)
  img = Image.fromarray(arr.astype(np.uint8), "RGBA")
/var/folders/_b/cfj7mly10r9f3nkywrlqnl300000gn/T/ipykernel_1765/1337777759.py:129: DeprecationWarning: 'mode' parameter is deprecated and will be removed in Pillow 13 (2026-10-15)
  base.paste(Image.fromarray(arr.astype(np.uint8), "RGBA"))
/var/folders/_b/cfj7mly10r9f3nkywrlqnl300000gn/T/ipykernel_1765/1337777759.py:68: DeprecationWarning: 'mode' parameter is deprecated and will be removed in Pillow 13 (2026-10-15)
  return Image.fromarray(arr.astype(np.uint8), "RGBA")
/var/folders/_b/cfj7mly10r9f3nkywrlqnl300000gn/T/ipykernel_1765/644622462.py:235: DeprecationWarning: 'mode' parameter is deprecated and will be removed in Pillow 13 (2026-10-15)
  img = Image.fromarray(arr.astype(np.uint8), "RGBA")


[RENDER] frame 24/192
[RENDER] frame 48/192
[RENDER] frame 72/192
[RENDER] frame 96/192
[RENDER] frame 120/192
[RENDER] frame 144/192
[RENDER] frame 168/192

[SAVED] /Users/mloktionov/PycharmProjects/Stellar_Attractor/ANIM/Infographics/Telemetry/animations/solar_coronal_loops/solar_coronal_loops.gif


This visualization shows an active solar region threaded by gigantic magnetic loops rising high above the photosphere. Streams of hot plasma race along the magnetic field lines while the corona continuously shifts and reconnects under extreme magnetic stress. Near the end of the sequence, part of the magnetic structure becomes unstable and erupts outward as a coronal mass ejection (CME) — a colossal cloud of magnetized plasma expelled into interplanetary space. The animation illustrates the layered structure of the Sun’s atmosphere: dense opaque photosphere below, glowing coronal arches above, and expanding shock fronts propagating through the solar environment.

# Gravitational lensing caustics / Einstein ring dynamics

In [33]:
from __future__ import annotations

from pathlib import Path
import numpy as np
import imageio.v2 as imageio
from PIL import Image, ImageDraw, ImageFilter


# ============================================================
# Einstein Ring Dynamics
# Gravitational lensing visualization
# ============================================================

OUTPUT_FORMAT = "gif"   # gif | mp4
FPS = 24
DURATION = 8
TOTAL_FRAMES = FPS * DURATION

WIDTH = 960
HEIGHT = 540

ANIMATION_NAME = "einstein_ring_dynamics"

OUT_DIR = Path("media-site/animations") / ANIMATION_NAME
OUT_DIR.mkdir(parents=True, exist_ok=True)

OUT_FILE = OUT_DIR / f"{ANIMATION_NAME}.{OUTPUT_FORMAT}"

BG = (0, 0, 0, 255)

RNG_SEED = 42

LENS_RADIUS = 34
EINSTEIN_RADIUS = 128

STAR_COUNT = 500

CAMERA_DRIFT = 22

rng = np.random.default_rng(RNG_SEED)


# ============================================================
# STARFIELD
# ============================================================

stars = []

for _ in range(STAR_COUNT):
    x = rng.uniform(0, WIDTH)
    y = rng.uniform(0, HEIGHT)

    size = rng.uniform(0.4, 2.0)
    brightness = rng.uniform(45, 210)

    temp = rng.uniform(0, 1)
    twinkle_phase = rng.uniform(0, 2 * np.pi)
    twinkle_speed = rng.uniform(0.35, 1.15)

    if temp < 0.33:
        color = (
            140 + rng.integers(0, 40),
            180 + rng.integers(0, 50),
            255,
        )
    elif temp < 0.66:
        color = (
            255,
            230 + rng.integers(0, 20),
            180 + rng.integers(0, 30),
        )
    else:
        color = (
            255,
            180 + rng.integers(0, 40),
            120 + rng.integers(0, 40),
        )

    stars.append((x, y, size, brightness, color, twinkle_phase, twinkle_speed))


# ============================================================
# HELPERS
# ============================================================

def rgba():
    return Image.new("RGBA", (WIDTH, HEIGHT), (0, 0, 0, 0))


def add_glow(base, layer, blur=8):
    glow = layer.filter(ImageFilter.GaussianBlur(blur))
    base.alpha_composite(glow)
    base.alpha_composite(layer)


def smoothstep(x):
    x = np.clip(x, 0, 1)
    return x * x * (3 - 2 * x)


# ============================================================
# LENS DISTORTION
# ============================================================

def distort_point(x, y, cx, cy):

    dx = x - cx
    dy = y - cy

    r = np.sqrt(dx * dx + dy * dy) + 1e-5

    theta = np.arctan2(dy, dx)

    # gravitational deflection
    alpha = (
        EINSTEIN_RADIUS ** 2
        / (r + 12)
    )

    # stronger near Einstein radius
    caustic = np.exp(
        -((r - EINSTEIN_RADIUS) ** 2)
        / (2 * 18 ** 2)
    )

    magnification = 1.0 + 3.8 * caustic

    warped_r = r + alpha * 0.12

    x2 = cx + warped_r * np.cos(theta)
    y2 = cy + warped_r * np.sin(theta)

    return x2, y2, magnification


# ============================================================
# BACKGROUND GALAXY
# ============================================================

def draw_background(base, phase):

    layer = rgba()
    d = ImageDraw.Draw(layer)

    drift_x = CAMERA_DRIFT * np.sin(phase * 2 * np.pi * 0.35)

    cx = WIDTH * 0.54 + drift_x
    cy = HEIGHT * 0.50

    # background stars through lens
    for i, (x, y, size, brightness, color, twinkle_phase, twinkle_speed) in enumerate(stars):
        twinkle = 0.12 + 0.88 * (
            0.5 + 0.5 * np.sin(
                phase * 2 * np.pi * twinkle_speed + twinkle_phase
            )
        ) ** 1.8

        x2, y2, mag = distort_point(x, y, cx, cy)

        rr = size * (
            1.0 + 0.8 * (mag - 1.0)
        )

        alpha = int(
            brightness
            * twinkle
            * min(mag, 4.0)
        )

        d.ellipse(
            [
                x2 - rr,
                y2 - rr,
                x2 + rr,
                y2 + rr,
            ],
            fill=(
                color[0],
                color[1],
                color[2],
                np.clip(alpha, 0, 255),
            ),
        )

    # Einstein ring
    ring_t = (
        0.5
        + 0.5 * np.sin(phase * 2 * np.pi)
    )

    ring_alpha = int(60 + 90 * ring_t)

    for width, alpha_mul in [
        (26, 0.15),
        (14, 0.35),
        (5, 1.0),
    ]:

        d.ellipse(
            [
                cx - EINSTEIN_RADIUS,
                cy - EINSTEIN_RADIUS,
                cx + EINSTEIN_RADIUS,
                cy + EINSTEIN_RADIUS,
            ],
            outline=(
                120,
                220,
                255,
                int(ring_alpha * alpha_mul),
            ),
            width=width,
        )

    # central lens object
    rr = LENS_RADIUS

    for scale, alpha in [
        (4.0, 10),
        (2.4, 24),
        (1.5, 55),
    ]:

        r2 = rr * scale

        d.ellipse(
            [
                cx - r2,
                cy - r2,
                cx + r2,
                cy + r2,
            ],
            fill=(120, 220, 255, alpha),
        )

    d.ellipse(
        [
            cx - rr,
            cy - rr,
            cx + rr,
            cy + rr,
        ],
        fill=(2, 2, 4, 255),
        outline=(220, 250, 255, 180),
        width=2,
    )

    add_glow(base, layer, blur=10)


# ============================================================
# CAUSTIC ARCS
# ============================================================

def draw_caustics(base, phase):

    layer = rgba()
    d = ImageDraw.Draw(layer)

    drift_x = CAMERA_DRIFT * np.sin(phase * 2 * np.pi * 0.35)

    cx = WIDTH * 0.54 + drift_x
    cy = HEIGHT * 0.50

    for k in range(4):

        t = np.linspace(-1.1, 1.1, 300)

        radius = (
            EINSTEIN_RADIUS
            + 8 * np.sin(phase * 2 * np.pi * 2 + k)
        )

        theta0 = (
            phase * 2 * np.pi * 0.7
            + k * np.pi / 2
        )

        theta = theta0 + t * 0.42

        deform = (
            1
            + 0.08 * np.sin(theta * 7 + phase * 9)
        )

        x = cx + radius * np.cos(theta) * deform
        y = cy + radius * np.sin(theta) * deform

        pts = list(zip(x, y))

        pulse = (
            0.5
            + 0.5 * np.sin(
                phase * 2 * np.pi * 3
                + k
            )
        )

        alpha = int(55 + 120 * pulse)

        d.line(
            pts,
            fill=(160, 240, 255, alpha),
            width=3,
            joint="curve",
        )

    add_glow(base, layer, blur=8)


# ============================================================
# SHIMMER FIELD
# ============================================================

def draw_shimmer(base, phase):

    layer = rgba()
    d = ImageDraw.Draw(layer)

    drift_x = CAMERA_DRIFT * np.sin(phase * 2 * np.pi * 0.35)

    cx = WIDTH * 0.54 + drift_x
    cy = HEIGHT * 0.50

    for i in range(120):

        theta = rng.uniform(0, 2 * np.pi)

        rr = (
            EINSTEIN_RADIUS
            + rng.normal(0, 12)
        )

        x = cx + rr * np.cos(theta)
        y = cy + rr * np.sin(theta)

        flicker = (
            0.5
            + 0.5 * np.sin(
                phase * 2 * np.pi * 6
                + i
            )
        )

        alpha = int(40 * flicker)

        r2 = rng.uniform(1, 3)

        d.ellipse(
            [x-r2, y-r2, x+r2, y+r2],
            fill=(180, 255, 255, alpha),
        )

    add_glow(base, layer, blur=4)


# ============================================================
# FRAME
# ============================================================

def render_frame(i):

    phase = i / TOTAL_FRAMES

    base = Image.new("RGBA", (WIDTH, HEIGHT), BG)

    draw_background(base, phase)

    draw_caustics(base, phase)

    draw_shimmer(base, phase)

    return np.array(base.convert("RGB"))


# ============================================================
# RENDER
# ============================================================

frames = []

print("[START] Einstein Ring Dynamics")

for i in range(TOTAL_FRAMES):

    if i % FPS == 0:
        print(f"[RENDER] frame {i}/{TOTAL_FRAMES}")

    frames.append(render_frame(i))

if OUTPUT_FORMAT == "gif":

    imageio.mimsave(
        OUT_FILE,
        frames,
        fps=FPS,
        loop=0,
    )

else:

    imageio.mimsave(
        OUT_FILE,
        frames,
        fps=FPS,
        quality=9,
        macro_block_size=1,
    )

print()
print(f"[SAVED] {OUT_FILE.resolve()}")

[START] Einstein Ring Dynamics
[RENDER] frame 0/192
[RENDER] frame 24/192
[RENDER] frame 48/192
[RENDER] frame 72/192
[RENDER] frame 96/192
[RENDER] frame 120/192
[RENDER] frame 144/192
[RENDER] frame 168/192

[SAVED] /Users/mloktionov/PycharmProjects/Stellar_Attractor/ANIM/Infographics/Telemetry/animations/einstein_ring_dynamics/einstein_ring_dynamics.gif


In [34]:
from __future__ import annotations

from pathlib import Path
import numpy as np
import imageio.v2 as imageio
from PIL import Image, ImageDraw, ImageFilter


# ============================================================
# Earth Magnetosphere Impact
# CME / solar wind compression + aurora response
# ============================================================

OUTPUT_FORMAT = "gif"  # gif | mp4
FPS = 24
DURATION = 8
TOTAL_FRAMES = FPS * DURATION

WIDTH = 960
HEIGHT = 540

ANIMATION_NAME = "earth_magnetosphere_impact"

OUT_DIR = Path("media-site/animations") / ANIMATION_NAME
OUT_DIR.mkdir(parents=True, exist_ok=True)
OUT_FILE = OUT_DIR / f"{ANIMATION_NAME}.{OUTPUT_FORMAT}"

BG = (0, 0, 0, 255)

RNG_SEED = 42
rng = np.random.default_rng(RNG_SEED)

EARTH_X = WIDTH * 0.58
EARTH_Y = HEIGHT * 0.52
EARTH_R = 74

BOW_SHOCK_X = EARTH_X - 165

FIELD_LINE_COUNT = 18
SOLAR_WIND_PARTICLES = 1200

IMPACT_PHASE = 0.48


# ============================================================
# HELPERS
# ============================================================

def rgba():
    return Image.new("RGBA", (WIDTH, HEIGHT), (0, 0, 0, 0))


def add_glow(base, layer, blur=8):
    glow = layer.filter(ImageFilter.GaussianBlur(blur))
    base.alpha_composite(glow)
    base.alpha_composite(layer)


def smoothstep(t):
    t = np.clip(t, 0.0, 1.0)
    return t * t * (3 - 2 * t)


# ============================================================
# BACKGROUND
# ============================================================

stars = []
for _ in range(650):
    stars.append((
        rng.uniform(0, WIDTH),
        rng.uniform(0, HEIGHT),
        rng.uniform(0.4, 1.5),
        rng.uniform(20, 110),
        rng.uniform(0.4, 1.2),
        rng.uniform(0, 2 * np.pi),
    ))


def draw_background(base, phase):
    d = ImageDraw.Draw(base)

    for x, y, r, a, speed, ph in stars:
        tw = 0.55 + 0.45 * np.sin(phase * 2 * np.pi * speed + ph) ** 2
        d.ellipse(
            [x - r, y - r, x + r, y + r],
            fill=(180, 220, 255, int(a * tw)),
        )


# ============================================================
# SOLAR WIND
# ============================================================

wind_particles = []
for _ in range(SOLAR_WIND_PARTICLES):
    wind_particles.append((
        rng.uniform(-WIDTH, WIDTH),
        rng.uniform(0, HEIGHT),
        rng.uniform(0.25, 1.1),
        rng.uniform(0.5, 2.0),
        rng.uniform(0.35, 1.0),
    ))


def draw_solar_wind(base, phase):
    layer = rgba()
    d = ImageDraw.Draw(layer)

    storm = smoothstep((phase - IMPACT_PHASE) / 0.18)
    speed_boost = 1.0 + 1.9 * storm

    for x0, y0, speed, size, alpha_mul in wind_particles:
        x = (x0 + phase * WIDTH * 1.8 * speed * speed_boost) % (WIDTH + 260) - 160
        y = y0 + 8 * np.sin(phase * 2 * np.pi * speed + x0 * 0.01)

        if x > EARTH_X - 40:
            continue

        tail = 18 + 34 * storm

        alpha = int((36 + 70 * storm) * alpha_mul)

        d.line(
            [(x - tail, y), (x, y)],
            fill=(110, 220, 255, alpha),
            width=max(1, int(size)),
        )

    add_glow(base, layer, blur=4)


# ============================================================
# MAGNETIC FIELD
# ============================================================

def magnetosphere_point(theta, scale, compression, tail_stretch):
    # dipole-like loop in screen coordinates
    r = EARTH_R * scale * (np.sin(theta) ** 2 + 0.18)

    x = EARTH_X + r * np.cos(theta)
    y = EARTH_Y + r * np.sin(theta)

    # dayside compression, nightside tail stretching
    if x < EARTH_X:
        x = EARTH_X + (x - EARTH_X) * compression
    else:
        x = EARTH_X + (x - EARTH_X) * tail_stretch

    return x, y


def draw_magnetic_field(base, phase):
    layer = rgba()
    d = ImageDraw.Draw(layer)

    storm = smoothstep((phase - IMPACT_PHASE) / 0.22)
    compression = 1.0 - 0.38 * storm
    tail_stretch = 1.0 + 1.3 * storm

    for i in range(FIELD_LINE_COUNT):
        scale = 1.6 + i * 0.18

        for side in [-1, 1]:
            t = np.linspace(0.12, np.pi - 0.12, 260)
            theta = t if side > 0 else -t

            pts = [magnetosphere_point(th, scale, compression, tail_stretch) for th in theta]

            alpha = int(38 + 72 * (1 - i / FIELD_LINE_COUNT))
            width = 1 if i > 4 else 2

            d.line(
                pts,
                fill=(90, 230, 255, alpha),
                width=width,
                joint="curve",
            )

    add_glow(base, layer, blur=6)


# ============================================================
# BOW SHOCK / CME FRONT
# ============================================================

def draw_bow_shock(base, phase):
    layer = rgba()
    d = ImageDraw.Draw(layer)

    storm = smoothstep((phase - IMPACT_PHASE) / 0.16)
    pulse = 0.5 + 0.5 * np.sin(phase * 2 * np.pi * 4) ** 2

    x = BOW_SHOCK_X - 42 * storm

    t = np.linspace(-1.25, 1.25, 420)

    width = 88 + 26 * storm
    height = 230 + 30 * storm

    bx = x + 42 * np.cos(t) ** 2
    by = EARTH_Y + height * np.sin(t) * 0.52

    pts = list(zip(bx, by))

    for w, a in [(18, 12), (9, 30), (3, int(95 + 55 * storm * pulse))]:
        d.line(
            pts,
            fill=(150, 245, 255, a),
            width=w,
            joint="curve",
        )

    add_glow(base, layer, blur=10)


def draw_cme_front(base, phase):
    storm_t = smoothstep((phase - 0.30) / 0.25)

    if storm_t <= 0:
        return

    layer = rgba()
    d = ImageDraw.Draw(layer)

    front_x = -160 + storm_t * (BOW_SHOCK_X + 90)

    t = np.linspace(-1.25, 1.25, 360)
    x = front_x + 36 * np.cos(t) ** 2
    y = EARTH_Y + 280 * np.sin(t) * 0.52

    pts = list(zip(x, y))

    alpha = int(95 * (1.0 - abs(storm_t - 0.75)))

    if alpha > 0:
        for w, a in [(28, 10), (14, 24), (4, alpha)]:
            d.line(
                pts,
                fill=(180, 255, 255, a),
                width=w,
                joint="curve",
            )

    add_glow(base, layer, blur=14)


# ============================================================
# EARTH / AURORA
# ============================================================

def draw_earth(base, phase):
    layer = rgba()
    d = ImageDraw.Draw(layer)

    cx, cy, r = EARTH_X, EARTH_Y, EARTH_R

    yy, xx = np.mgrid[0:HEIGHT, 0:WIDTH]
    dx = xx - cx
    dy = yy - cy
    rr = np.sqrt(dx * dx + dy * dy)

    sphere = rr <= r
    shade = np.clip(1.0 - rr / r, 0, 1)

    texture = (
        0.52
        + 0.22 * np.sin(dx * 0.055 + phase * 2)
        + 0.17 * np.sin(dy * 0.065 - phase * 1.4)
        + 0.10 * np.sin((dx + dy) * 0.035)
    )
    texture = np.clip(texture, 0, 1)

    arr = np.zeros((HEIGHT, WIDTH, 4), dtype=np.uint8)

    arr[..., 0] = np.where(sphere, 30 + 20 * texture, 0)
    arr[..., 1] = np.where(sphere, 75 + 90 * texture, 0)
    arr[..., 2] = np.where(sphere, 135 + 90 * shade, 0)
    arr[..., 3] = np.where(sphere, 255, 0)

    img = Image.fromarray(arr.astype(np.uint8), "RGBA")
    layer.alpha_composite(img)

    # atmosphere glow
    for scale, alpha in [(1.22, 30), (1.42, 12)]:
        d.ellipse(
            [
                cx - r * scale,
                cy - r * scale,
                cx + r * scale,
                cy + r * scale,
            ],
            outline=(100, 220, 255, alpha),
            width=3,
        )

    # aurora ovals
    storm = smoothstep((phase - IMPACT_PHASE) / 0.18)
    aurora_alpha = int(40 + 155 * storm)

    for sign in [-1, 1]:
        ay = cy + sign * r * 0.48
        d.ellipse(
            [
                cx - r * 0.62,
                ay - r * 0.16,
                cx + r * 0.62,
                ay + r * 0.16,
            ],
            outline=(80, 255, 160, aurora_alpha),
            width=3,
        )

    add_glow(base, layer, blur=6)


# ============================================================
# PARTICLE PRECIPITATION
# ============================================================

precip_particles = []
for _ in range(260):
    precip_particles.append((
        rng.uniform(0, 2 * np.pi),
        rng.uniform(0.0, 1.0),
        rng.choice([-1, 1]),
        rng.uniform(0.4, 1.2),
    ))


def draw_precipitation(base, phase):
    storm = smoothstep((phase - IMPACT_PHASE) / 0.15)

    if storm <= 0.02:
        return

    layer = rgba()
    d = ImageDraw.Draw(layer)

    for angle, offset, sign, speed in precip_particles:
        t = (offset + phase * speed * 2.2) % 1.0

        x0 = EARTH_X - 180 + 210 * t
        y0 = EARTH_Y + sign * (160 - 95 * t) + 18 * np.sin(angle + phase * 8)

        x1 = EARTH_X + sign * 12 + 38 * np.cos(angle)
        y1 = EARTH_Y + sign * EARTH_R * 0.48

        x = x0 * (1 - t) + x1 * t
        y = y0 * (1 - t) + y1 * t

        alpha = int(130 * storm * (1 - abs(t - 0.65)))

        if alpha > 0:
            d.ellipse(
                [x - 1.4, y - 1.4, x + 1.4, y + 1.4],
                fill=(180, 255, 220, alpha),
            )

    add_glow(base, layer, blur=5)


# ============================================================
# FRAME
# ============================================================

def render_frame(i):
    phase = i / TOTAL_FRAMES

    base = Image.new("RGBA", (WIDTH, HEIGHT), BG)

    draw_background(base, phase)
    draw_solar_wind(base, phase)
    draw_cme_front(base, phase)
    draw_magnetic_field(base, phase)
    draw_bow_shock(base, phase)
    draw_precipitation(base, phase)
    draw_earth(base, phase)

    return np.array(base.convert("RGB"))


# ============================================================
# RENDER
# ============================================================

frames = []

print("[START] Earth Magnetosphere Impact")

for i in range(TOTAL_FRAMES):
    if i % FPS == 0:
        print(f"[RENDER] frame {i}/{TOTAL_FRAMES}")
    frames.append(render_frame(i))

if OUTPUT_FORMAT == "gif":
    imageio.mimsave(
        OUT_FILE,
        frames,
        fps=FPS,
        loop=0,
    )
else:
    imageio.mimsave(
        OUT_FILE,
        frames,
        fps=FPS,
        quality=9,
        macro_block_size=1,
    )

print()
print(f"[SAVED] {OUT_FILE.resolve()}")

[START] Earth Magnetosphere Impact
[RENDER] frame 0/192


/var/folders/_b/cfj7mly10r9f3nkywrlqnl300000gn/T/ipykernel_1765/607624582.py:281: DeprecationWarning: 'mode' parameter is deprecated and will be removed in Pillow 13 (2026-10-15)
  img = Image.fromarray(arr.astype(np.uint8), "RGBA")


[RENDER] frame 24/192
[RENDER] frame 48/192
[RENDER] frame 72/192
[RENDER] frame 96/192
[RENDER] frame 120/192
[RENDER] frame 144/192
[RENDER] frame 168/192

[SAVED] /Users/mloktionov/PycharmProjects/Stellar_Attractor/ANIM/Infographics/Telemetry/animations/earth_magnetosphere_impact/earth_magnetosphere_impact.gif
